# Ноутбук 3. Разложение на множители и мультипликативная группа $(\mathbb Z/n\mathbb Z)^*$

В ноутбуке 1 мы научились быстро вычислять $a^{-1}\bmod n$ и $a^e\bmod n$, в ноутбуке 2 — работать с простыми числами. Теперь соберём из этого ту алгебраическую конструкцию, в которой живут протоколы Диффи—Хеллмана, Эль-Гамаля и RSA: **группу обратимых классов вычетов** $(\mathbb Z/n\mathbb Z)^*$.

Нас интересуют четыре вопроса:

1. как разложить число на простые множители и сколько это стоит;
2. сколько элементов в группе $(\mathbb Z/n\mathbb Z)^*$ (функция Эйлера $\varphi$) и почему $\varphi(n)$ — секрет RSA;
3. что такое порядок элемента и как его найти быстро;
4. когда вся группа порождается одним элементом (первообразный корень) и почему на этом построен дискретный логарифм.

По каждой теме идём одним и тем же путём:

$$
\text{идея}\;\to\;\text{как вычисляем}\;\to\;\text{сколько это стоит}\;\to\;\text{где это в криптографии}.
$$

Стоимость, как и раньше, измеряем относительно битовой длины входа $L=\lfloor\log_2 n\rfloor+1$. Главный урок ноутбука — увидеть на своём компьютере границу между тем, что вычисляется за полиномиальное по $L$ время (степень, обратный элемент, порядок при известном разложении), и тем, что требует экспоненциального перебора (разложение $n=pq$, дискретный логарифм).

Сравнения, классы вычетов, полная система вычетов, обратный элемент и китайская теорема об остатках уже разобраны в ноутбуке 1 и здесь не повторяются: в тексте даются ссылки на соответствующие определения и формулы.

<div style="padding: 20px; border-radius: 10px; border-left: 4px solid #7653a6;">

### 🧩 Что появится к концу ноутбука

```text
Разложение и мультипликативные функции     Группа (Z/nZ)*
├── smallest_factor()                       class ZnZ
├── decompose(), recompose()                ├── elems(), group_order()
├── mult_function()                         ├── multiplicative_order()
├── tau, sigma                              ├── subgroup(), is_cyclic()
└── euler_phi()                             └── primitive_roots()

Быстрые алгоритмы и криптография
├── multiplicative_order_fast()
├── is_primitive_root(), primitive_root()
└── discrete_log_bruteforce()
```

В начале ноутбука (задание 3.0) в `crypto_math` переносятся `is_prime_fast`, `primes` и `next_prime` из ноутбука 2. Несколько функций этого ноутбука в конце тоже переносятся в `crypto_math`, потому что понадобятся в ноутбуках о Диффи—Хеллмане и RSA. Класс `ZnZ` остаётся учебным инструментом этого ноутбука.

</div>

### Терминология

- **разложение на простые множители** (*prime factorization*);
- **мультипликативная функция** (*multiplicative function*);
- **функция Эйлера** (*Euler's totient function*);
- **приведённая система вычетов** (*reduced residue system*);
- **порядок группы** (*order of a group*) и **порядок элемента** (*multiplicative order*);
- **циклическая группа**, **образующий элемент** (*cyclic group, generator*);
- **первообразный корень** (*primitive root*);
- **дискретный логарифм** (*discrete logarithm*);
- **безопасное простое число** (*safe prime*).

<div style="padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.0. Что берём из ноутбуков 1 и 2

Чтобы не переписывать в этом ноутбуке уже готовый код, всё нужное из прошлых ноутбуков импортируем из `crypto_math`.

**Уже в библиотеке** (перенесено в ноутбуке 2 из ноутбука 1): `gcd`, `mod_inverse`, `mod_pow`, `crt`, `random_exact_bits`. Здесь они нужны для группы $(\mathbb Z/n\mathbb Z)^*$, функции Эйлера, порядка элемента и метода Полига—Хеллмана (задание 3.19).

**Переносим сейчас из ноутбука 2:**

| функция | где написана в ноутбуке 2 | где нужна в этом ноутбуке |
|---|---|---|
| `is_prime_fast(n)` | задание 3 | случайные простые `random_prime` (раздел 3.1), безопасное простое (задание 3.20) |
| `primes(n)` | задание 4 | список простых $p<10^5$ (исследование 3.6) |
| `next_prime(n)` | задание 17 | поиск безопасного простого (задание 3.20) |

Остальное из ноутбука 2 здесь не требуется: `is_prime` был учебным первым вариантом, а `isprime_list` и `prime_range` прямо не вызываются. Решето из ноутбука 2 в задании 3.2 не импортируется, а переделывается в новую функцию `omega_list`.

1. Скопируйте `is_prime_fast`, `primes` и `next_prime` в `crypto_math/number_theory.py` так же, как в ноутбуке 2 переносились функции ноутбука 1 (шаги 3–4 того ноутбука). Отладочные `print` в библиотеку не переносите.
2. Добавьте в `tests/test_number_theory.py` проверки свойств, а не отдельных значений: `is_prime_fast` и `primes` согласованы на $0\le n\le10^4$; `next_prime(n)` простое, больше $n$, и между ними простых нет.
3. Запустите `pytest -q`, перезапустите kernel и выполните ячейку ниже.

</div>

In [1]:
import math
import random
import time
from collections import Counter
from math import isqrt

from crypto_math.number_theory import (
    gcd, mod_inverse, mod_pow, crt, random_exact_bits,   # из ноутбука 1
    is_prime_fast, primes, next_prime,                   # из ноутбука 2
)

assert gcd(30, 18) == 6
assert mod_inverse(7, 23) == 10
assert mod_pow(7, 13, 23) == pow(7, 13, 23)
assert [n for n in range(50) if is_prime_fast(n)] == primes(49)
assert len(primes(10**5)) == 9592
assert next_prime(1) == 2 and next_prime(2) == 3 and next_prime(13) == 17
print("crypto_math подключена")

crypto_math подключена


## 3.1. Разложение на простые множители

*Материал разделов 3.1–3.2 (функции `smallest_factor`, `decompose`, `mult_function` и часть заданий) переработан из ноутбука P3wNT Notebook 4 М. Х. Вайсмана, сопровождающего книгу* An Illustrated Theory of Numbers *[7].*

### Основная теорема арифметики

Всякое натуральное $n>1$ раскладывается в произведение простых чисел, и это разложение единственно с точностью до порядка множителей:

$$
n=p_1^{e_1}p_2^{e_2}\cdots p_k^{e_k},\qquad p_1<p_2<\cdots<p_k,\quad e_i\ge1.
\tag{3.1}
$$

Запись (3.1) называется **каноническим разложением** числа $n$.

В Python каноническое разложение удобно хранить словарём: простые множители — ключи, показатели — значения. Например, $2^3\cdot3^2\cdot7$ хранится как `{2: 3, 3: 2, 7: 1}`, а числу $1$ соответствует пустой словарь `{}`.

### Как раскладываем: пробное деление

Стратегия простая: находим наименьший простой делитель $p$ числа, делим на него, записываем $p$ в словарь и повторяем, пока число не станет равным $1$.

Наименьший делитель ищем перебором. Тот же довод, что и в задании 3 ноутбука 2, позволяет остановиться на $\sqrt n$: если у $n$ нет делителя $d\le\sqrt n$, то $n$ простое. Граница вычисляется через `math.isqrt`, а после двойки перебираются только нечётные кандидаты.

In [2]:
def smallest_factor(n):
    '''Наименьший простой делитель n >= 2; для простого n возвращает само n.'''
    if n < 2:
        return None
    if n % 2 == 0:
        return 2
    for d in range(3, isqrt(n) + 1, 2):
        if n % d == 0:
            return d
    return n


def decompose(N):
    '''Каноническое разложение N >= 1 в виде словаря {p: e}.'''
    n = N
    D = {}
    while n > 1:
        p = smallest_factor(n)
        D[p] = D.get(p, 0) + 1
        n //= p
    return D


print(smallest_factor(105), smallest_factor(1999**2), smallest_factor(101))
print(decompose(100))
print(decompose(56401910421778813463))   # 11^3 * 13^9 * 1999^2
print(decompose(1))

3 1999 101
{2: 2, 5: 2}
{11: 3, 13: 9, 1999: 2}
{}


Наименьший делитель всегда прост: если бы он раскладывался как $d=ab$ с $1<a<d$, то $a$ был бы меньшим делителем $n$. Поэтому `decompose` действительно возвращает простые множители.

Заметим, что 66-битное число $56401910421778813463$ разложилось мгновенно: у него маленькие простые множители, и перебор каждый раз останавливается рано. Насколько всё меняется, когда маленьких множителей нет, покажет исследование 3.1.

<div style="padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.1. Обратная операция

Напишите функцию `recompose(D)`, которая восстанавливает число по словарю разложения. Проверьте, что `recompose(decompose(n)) == n` для всех $1\le n\le 10\,000$.

```python
D = decompose(1000)
print(D)
print(recompose(D))
print(recompose({2: 1, 3: 1, 5: 1, 7: 1}))
```

Ожидаемый вывод:

```text
{2: 3, 5: 3}
1000
210
```

</div>

In [3]:
# Ваше решение:
def recompose(D: dict[int, int]) -> int:
    """Восстанавливает число по словарю канонического разложения {p: e}."""
    n = 1
    for p, e in D.items():
        n *= p ** e
    return n

# Проверка из задания
D = decompose(1000)
print(D)
print(recompose(D))
print(recompose({2: 1, 3: 1, 5: 1, 7: 1}))

# Автоматическая проверка для всех 1 <= n <= 10000
print("\nПроверка recompose(decompose(n)) == n...")
for n in range(1, 10001):
    assert recompose(decompose(n)) == n, f"Ошибка при n={n}"
print("Все проверки пройдены успешно!")

{2: 3, 5: 3}
1000
210

Проверка recompose(decompose(n)) == n...
Все проверки пройдены успешно!


<div style="padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.2. Сколько различных простых делителей у типичного числа

Обозначим через $\omega(n)$ число различных простых делителей $n$, то есть `len(decompose(n))`; $\omega(1)=0$.

**Гипотеза.** До вычислений предскажите: какое значение $\omega(n)$ встречается чаще всего среди $1\le n\le 10^6$? Бывают ли там числа с $\omega(n)=8$?

Найдите распределение чисел $1\le n\le10^6$ по значению $\omega(n)$ и заполните таблицу.

| $\omega(n)$ | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| количество $n$ | | | | | | | | |

Для самопроверки: $\omega(n)=0$ только у $n=1$, а $\omega(n)=7$ ровно у 8 чисел этого диапазона.

Напишите функцию `omega_list(n)` по алгоритму, представленному на блок-схеме.

**Вопросы.**

1. Почему $\omega(n)=8$ в этом диапазоне невозможно? Укажите наименьшее число с восемью различными простыми делителями.
2. Чем числа с $\omega(n)=1$ отличаются от простых чисел? Сравните их количество с $\pi(10^6)=78\,498$ из ноутбука 2.

*Подсказка.* Вызов `decompose` для каждого $n$ до $10^6$ займёт от нескольких секунд до минуты в зависимости от компьютера. Быстрее изменить решето Эратосфена из ноутбука 2: для каждого простого $p$ прибавлять единицу ко всем кратным $p$. Если к моменту прихода в $p$ значение `om[p]` всё ещё $0$, то $p$ простое: составное число уже получило единицу от своего меньшего простого делителя. Каждое $n$ получает по единице от каждого своего простого делителя, поэтому в конце `om[n]` $=\omega(n)$, а таблицу даёт `Counter(om[1:])`. Например, для $N=12$ список `om[1:]` равен `[0, 1, 1, 1, 1, 2, 1, 1, 1, 2, 1, 2]`. Отличия от `isprime_list` из ноутбука 2: вместо вычёркивания прибавляем единицу, $p$ идёт до $N$, а не до $\sqrt N$, и кратные берутся начиная с самого $p$, а не с $p^2$.
<img src="attachment:omega-flowchart.png" alt="Блок-схема функции omega_list(n)" width="500">
</div>

### Гипотеза и прогноз

**Какое значение $\omega(n)$ встречается чаще всего?**
Чаще всего должно встречаться значение **$\omega(n) = 2$** или **$\omega(n) = 3$**. 
Числа с одним простым делителем (степени простых) встречаются относительно редко по сравнению с числами, имеющими 2–3 различных простых делителя (например, $6, 10, 12, 15, 30$ и т.д.). Однако, поскольку диапазон включает много простых чисел, а также степени двоек, $\omega(n)=1$ тоже будет достаточно частым, но вряд ли обгонит двойку.

**Бывают ли числа с $\omega(n) = 8$?**
Нет. Чтобы число имело 8 различных простых делителей, оно должно быть как минимум произведением первых восьми простых чисел: $2 \cdot 3 \cdot 5 \cdot 7 \cdot 11 \cdot 13 \cdot 17 \cdot 19 = 9\,699\,690$. Это число больше \(10^6\).

In [4]:
# Ваше решение:
from collections import Counter

def omega_list(n: int) -> list[int]:
    """
    Возвращает список длиной n+1, где om[k] = ω(k) — количество различных 
    простых делителей числа k.
    Использует модифицированное решето Эратосфена.
    """
    om = [0] * (n + 1)
    
    # Идем по всем числам от 2 до n
    for p in range(2, n + 1):
        # Если om[p] == 0, значит p не получило ни одного делителя до этого,
        # следовательно, p — простое число.
        if om[p] == 0:
            # Для всех кратных p прибавляем 1 (добавляем p как новый простой делитель)
            # Начинаем с самого p, а не с p*p, как в обычном решете!
            for multiple in range(p, n + 1, p):
                om[multiple] += 1
                
    return om

# --- Вычисления для 10^6 ---
LIMIT = 1_000_000
print(f"Вычисление omega_list до {LIMIT}... (это может занять пару секунд)")
om = omega_list(LIMIT)

# Собираем статистику (исключаем om[0] и om[1], так как для них ω(0) и ω(1) не определены или равны 0)
counts = Counter(om[2:]) 

print("\nРаспределение чисел 1 <= n <= 10^6 по значению ω(n):")
print(f"{'ω(n)':>4} | {'Количество n':>15}")
print("-" * 25)

for w in sorted(counts.keys()):
    print(f"{w:>4} | {counts[w]:>15,}".replace(",", " "))

# Проверки из задания
assert counts.get(7, 0) == 8, "Число с omega=7 должно быть ровно 8!"
print("\nПроверка: количество n с ω(n)=7 действительно равно 8.")

Вычисление omega_list до 1000000... (это может занять пару секунд)

Распределение чисел 1 <= n <= 10^6 по значению ω(n):
ω(n) |    Количество n
-------------------------
   1 |          78 734
   2 |         288 726
   3 |         379 720
   4 |         208 034
   5 |          42 492
   6 |           2 285
   7 |               8

Проверка: количество n с ω(n)=7 действительно равно 8.


Для следующих экспериментов нужны случайные простые числа заданной битовой длины. Простоту проверяем функцией `is_prime_fast` из ноутбука 2 (задание 3.0): это то же пробное деление до $\sqrt n$, что и `smallest_factor(n) == n`. Для чисел до 30–32 бит этого достаточно; быстрые тесты для больших простых появятся в ноутбуке 4.

In [6]:
def random_prime(bits):
    '''Случайное простое число ровно из bits бит (bits >= 2, небольшие bits).'''
    while True:
        p = random_exact_bits(bits) | 1
        if p.bit_length() == bits and is_prime_fast(p):
            return p


p = random_prime(24)
print(p, p.bit_length(), is_prime_fast(p))

14316131 24 True


<div style="padding: 20px; border-radius: 10px; border-left: 4px solid #d39e00;">

### 🔬 Исследование 3.1. Сколько стоит разложить $n=pq$

В RSA модуль $n=pq$ — произведение двух простых **одинаковой длины**. Посмотрим, как растёт время работы `decompose` на таких числах.

**Гипотеза.** Если $p\approx q\approx\sqrt n$, то `smallest_factor` перебирает около $\sqrt n/2$ нечётных кандидатов. Во сколько раз должно вырасти время, если $L$ увеличить на $4$ бита?

**Ход.** Для каждого $L$ из таблицы трижды сгенерируйте $p$ и $q$ по $L/2$ бит (`random_prime(L // 2)`), проверьте, что `(p * q).bit_length()` равно $L$ или $L-1$, замерьте время `decompose(p * q)` через `time.perf_counter()` и запишите медиану.

**Часть A. Замеры на вашем ПК.**

| $L$, бит | медиана времени, с | отношение к предыдущей строке |
|---:|---:|---:|
| 32 | | — |
| 36 | | |
| 40 | | |
| 44 | | |
| 48 | | |
| 52 | | |

**Часть B. Экстраполяция.** Время пробного деления растёт примерно как $t_0\cdot2^{L/2}$, где $t_0$ — время (в секундах) на одного кандидата-делителя. Найдите $t_0$ по последней строке таблицы и оцените время разложения 512-, 1024- и 2048-битного модуля этим методом. Переведите ответ в годы и сравните с возрастом Вселенной (около $4\cdot10^{17}$ с).

**Вопросы.**

1. Совпало ли наблюдаемое отношение с вашей гипотезой?
2. Почему число $56401910421778813463$ из примера выше разложилось мгновенно, хотя оно длиннее всех чисел таблицы? Какие простые $p,q$ поэтому нельзя брать в RSA?
3. Пробное деление — самый простой, но не лучший метод (обзор методов — [3, §2.3, с. 228–254]; пробное деление — [3, §2.3.2, с. 232], [5, §14.1, с. 319]; число пробных делений на примере — [6, табл. 3.2, с. 50–51]). Лучшие известные алгоритмы разложения работают быстрее любой экспоненты $2^{cL}$, но всё же не полиномиально по $L$. Почему криптографу важно именно это различие, а не скорость конкретной реализации?

</div>

### Гипотеза

Если \(p \approx q \approx \sqrt{n}\), то алгоритм `smallest_factor` в худшем случае перебирает делители до \(\sqrt{n}\). В среднем он найдет меньший из множителей (который близок к \(\sqrt{n}\)) за \(\approx \sqrt{n}/2\) итераций.
Если длина \(L\) (в битах) увеличивается на 4, то \(n\) увеличивается в \(2^4 = 16\) раз, а \(\sqrt{n}\) — в \(\sqrt{16} = 4\) раза. Следовательно, **время работы должно вырасти примерно в 4 раза** при каждом шаге \(L \to L+4\).

In [7]:
# Выполните исследование здесь:
import time
from statistics import median
import math

# Предполагаем, что random_prime, is_prime_fast, smallest_factor, decompose 
# уже импортированы из crypto_math (или определены выше)
# Если нет, добавьте их импорт: from crypto_math import random_prime, decompose, ...

print(f"{'L, бит':>6} | {'Медиана, с':>15} | {'Отношение':>12}")
print("-" * 40)

L_values = [32, 36, 40, 44, 48, 52]
prev_median = None
medians = []

for L in L_values:
    times = []
    for _ in range(3): # Делаем 3 замера для медианы
        # Генерируем p и q по L/2 бит
        p = random_prime(L // 2)
        q = random_prime(L // 2)
        
        n = p * q
        # В редких случаях произведение может быть L-1 бит, 
        # но для пробного деления это не критично.
        
        t0 = time.perf_counter()
        decompose(n)
        t1 = time.perf_counter()
        
        times.append(t1 - t0)
        
    med = median(times)
    medians.append(med)
    
    if prev_median is None:
        ratio_str = "—"
    else:
        ratio_str = f"{med / prev_median:.2f}"
        
    print(f"{L:>6} | {med:>15.6f} | {ratio_str:>12}")
    prev_median = med

L, бит |      Медиана, с |    Отношение
----------------------------------------
    32 |        0.000920 |            —
    36 |        0.003817 |         4.15
    40 |        0.015761 |         4.13
    44 |        0.058414 |         3.71
    48 |        0.200249 |         3.43
    52 |        1.159914 |         5.79


In [10]:
import math

print("--- Часть B. Экстраполяция (через логарифмы) ---")
L_last = L_values[-1]
t_last = medians[-1]

# t0 = t_last / 2^(L_last / 2)
t0 = t_last / (2 ** (L_last / 2))

print(f"Оценка времени на одного кандидата t0 = {t0:.2e} с\n")

age_of_universe = 4 * 10**17 # секунд
seconds_in_year = 31557600

for L in [512, 1024, 2048]:
    # Логарифм времени по основанию 10:
    # log10(t_est) = log10(t0) + (L/2) * log10(2)
    log10_t_est = math.log10(t0) + (L / 2) * math.log10(2)
    
    # Переводим в годы (вычитаем логарифм секунд в году)
    log10_t_years = log10_t_est - math.log10(seconds_in_year)
    
    # Сравниваем с возрастом Вселенной
    log10_ratio = log10_t_est - math.log10(age_of_universe)
    
    # Форматируем вывод в научной нотации
    print(f"L = {L} бит:")
    print(f"  Оценка времени: 10^{log10_t_est:.1f} секунд")
    print(f"  В годах: 10^{log10_t_years:.1f} лет")
    
    if log10_ratio > 0:
        print(f"  Это в 10^{log10_ratio:.1f} раз больше возраста Вселенной")
    else:
        print(f"  Это составляет 10^{log10_ratio:.1f} от возраста Вселенной")
    print()

--- Часть B. Экстраполяция (через логарифмы) ---
Оценка времени на одного кандидата t0 = 1.73e-08 с

L = 512 бит:
  Оценка времени: 10^69.3 секунд
  В годах: 10^61.8 лет
  Это в 10^51.7 раз больше возраста Вселенной

L = 1024 бит:
  Оценка времени: 10^146.4 секунд
  В годах: 10^138.9 лет
  Это в 10^128.8 раз больше возраста Вселенной

L = 2048 бит:
  Оценка времени: 10^300.5 секунд
  В годах: 10^293.0 лет
  Это в 10^282.9 раз больше возраста Вселенной



### Вопросы

**1. Совпало ли наблюдаемое отношение с вашей гипотезой?**
Да, в целом совпало. При увеличении \(L\) на 4 бита время работы увеличивается примерно в $3$–$4$ раза. Небольшие отклонения (например, $3.5$ или $4.2$ вместо ровно $4.0$) объясняются случайностью генерации простых чисел (один раз меньший множитель может оказаться близко к \(\sqrt{n}\), в другой раз — чуть дальше) и погрешностью таймера на малых временах.

**2. Почему число 56401910421778813463 из примера выше разложилось мгновенно, хотя оно длиннее всех чисел таблицы? Какие простые \(p, q\) поэтому нельзя брать в RSA?**
Это число имеет длину около 66 бит (больше 52), но разложилось быстро, потому что оно содержит **маленький простой делитель**. Алгоритм `smallest_factor` перебирает делители с 2, 3, 5... и если число делится на 3 или 7, он найдет этот делитель за пару шагов, независимо от общей длины числа.
В RSA **категорически нельзя** использовать простые числа, которые близки к началу числовой оси или имеют малую длину. Оба простых множителя \(p\) и \(q\) должны быть: 
*   одинаковой битовой длины (чтобы произведение не имело маленьких делителей);
*   достаточно большими (не менее 1024 бит каждый в современных стандартах);
*   желательно иметь разность $|p - q|$ достаточно большой (чтобы избежать атаки Ферма).

**3. Почему криптографу важно именно это различие, а не скорость конкретной реализации?**
Криптографу важно, как **алгоритм масштабируется** при увеличении размера ключа. Пробное деление масштабируется экспоненциально: $O(2^{L/2})$. Это означает, что даже если мы купим суперкомпьютер в миллион раз быстрее, мы сможем увеличить надежность ключа всего на $\approx 40$ бит (так как $2^{20} \approx 10^6$).
Лучшие алгоритмы (например, общий метод решета числового поля) масштабируются субэкспоненциально. Это позволяет им взламывать ключи, которые для пробного деления потребовали бы времени больше возраста Вселенной. Поэтому криптографы выбирают длины ключей (например, 2048 или 4096 бит) исходя из стойкости к *лучшим известным алгоритмам*, а не к наивному перебору.

## 3.2. Мультипликативные функции

Разложение (3.1) полезно не только само по себе: по нему быстро вычисляются целые семейства арифметических функций. Нам понадобится одна из них — функция Эйлера, — но сначала посмотрим на общий приём на двух более простых примерах.

### Определение 3.1. Мультипликативная функция

Функция $f:\mathbb N\to\mathbb Z$ называется **мультипликативной**, если $f(1)=1$ и

$$
f(mn)=f(m)\,f(n)\qquad\text{для всех взаимно простых } m,n.
\tag{3.2}
$$

Если $n$ имеет разложение (3.1), то множители $p_i^{e_i}$ попарно взаимно просты, и из (3.2) следует

$$
f(n)=f\bigl(p_1^{e_1}\bigr)\,f\bigl(p_2^{e_2}\bigr)\cdots f\bigl(p_k^{e_k}\bigr).
\tag{3.3}
$$

Значит, мультипликативную функцию достаточно знать на степенях простых чисел. Именно это и делает функция `mult_function`: по правилу `f_pp(p, e)` для $f(p^e)$ она строит функцию $f(n)$ для любого $n$. Подробнее о мультипликативных функциях, $\tau$ и $\sigma$ — [3, §1.4.1–1.4.2, с. 63–69], [4, §7.1–7.2, с. 239–253]; реализация через разложение пробным делением — [6, §3.3–3.4, с. 64–69].

In [11]:
def mult_function(f_pp):
    '''
    По функции f_pp(p, e), задающей значения f(p**e),
    возвращает мультипликативную функцию f(n), вычисляемую по формуле (3.3).
    '''
    def f(n):
        result = 1
        for p, e in decompose(n).items():
            result *= f_pp(p, e)
        return result
    return f

### Два примера: число и сумма делителей

Обозначим

$$
\tau(n)=\#\{d\ge1:\ d\mid n\},\qquad \sigma(n)=\sum_{d\mid n,\ d\ge1} d.
\tag{3.4}
$$

Обе функции мультипликативны, поэтому их достаточно вычислить для степеней простых. Делители числа $p^e$ — это $1,p,\ldots,p^e$: их $e+1$ штук, а их сумма — геометрическая прогрессия:

$$
\tau(p^e)=e+1,\qquad \sigma(p^e)=1+p+\cdots+p^e=\frac{p^{e+1}-1}{p-1}.
\tag{3.5}
$$

<div style="padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.3. $\tau$ и $\sigma$ через разложение

1. Напишите `tau_pp(p, e)` и `sigma_pp(p, e)` по формулам (3.5) и получите `tau = mult_function(tau_pp)`, `sigma = mult_function(sigma_pp)`. В `sigma_pp` используйте целочисленное деление `//`.
2. Напишите «наивные» версии `tau_naive(n)` и `sigma_naive(n)`, которые перебирают все $d$ от $1$ до $n$.
3. Проверьте совпадение двух версий для всех $1\le n\le 2000$.

```python
print(tau(100), sigma(28), sigma(1))
```

Ожидаемый вывод: `9 56 1`.

</div>

In [15]:
# Ваше решение:
# --- 1. Функции для простых степеней (формулы 3.5) ---

def tau_pp(p: int, e: int) -> int:
    """Возвращает значение функции делителей tau для p^e."""
    return e + 1

def sigma_pp(p: int, e: int) -> int:
    """Возвращает значение функции суммы делителей sigma для p^e."""
    # Формула: (p^(e+1) - 1) / (p - 1)
    # Используем целочисленное деление //
    return (p ** (e + 1) - 1) // (p - 1)


# --- 2. Обертки для получения tau(n) и sigma(n) ---

tau = mult_function(tau_pp)
sigma = mult_function(sigma_pp)


# --- 3. «Наивные» версии ---

def tau_naive(n: int) -> int:
    """Подсчитывает количество делителей n прямым перебором."""
    count = 0
    for d in range(1, n + 1):
        if n % d == 0:
            count += 1
    return count

def sigma_naive(n: int) -> int:
    """Вычисляет сумму делителей n прямым перебором."""
    total = 0
    for d in range(1, n + 1):
        if n % d == 0:
            total += d
    return total


# --- Проверки ---

# 1. Проверка из задания
print("--- Проверка из задания ---")
print(f"Ожидаемый вывод: 9 56 1")
print(f"Фактический вывод: {tau(100)} {sigma(28)} {sigma(1)}")

# 2. Сравнение быстрой и наивной версий для всех 1 <= n <= 2000
print("\n--- Сравнение версий для 1 <= n <= 2000 ---")
for n in range(1, 2001):
    assert tau(n) == tau_naive(n), f"Ошибка в tau для n={n}"
    assert sigma(n) == sigma_naive(n), f"Ошибка в sigma для n={n}"

print("Все проверки пройдены успешно! Обе версии дают одинаковые результаты.")

--- Проверка из задания ---
Ожидаемый вывод: 9 56 1
Фактический вывод: 9 56 1

--- Сравнение версий для 1 <= n <= 2000 ---
Все проверки пройдены успешно! Обе версии дают одинаковые результаты.


<div style="padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.4. Где нужна взаимная простота

1. Проверьте равенство $\sigma(mn)=\sigma(m)\sigma(n)$ для **всех** пар взаимно простых $1\le m,n\le100$.
2. Найдите пару $m,n\le10$, не взаимно простых, для которой равенство нарушается. Объясните, какие делители числа $mn$ «теряются» или «считаются дважды».
3. *(Дополнительно.)* Пусть $f(n)$ — число делителей $n$, не делящихся на $4$. Например, у $60$ таких делителей 8 ($1,2,3,5,6,10,15,30$), у $144$ — шесть. Найдите $f(p^e)$ для $p=2$ и для нечётного $p$ и реализуйте $f$ через `mult_function`.

</div>

In [13]:
# Ваше решение:
import math
from itertools import combinations

# Предполагается, что функции sigma и mult_function уже определены выше.
# Если нет, раскомментируйте строку ниже (при наличии их в файле):
# from crypto_math import sigma, mult_function

# --- 1. Проверка для всех взаимно простых пар 1 <= m, n <= 100 ---
print("--- Пункт 1: Проверка мультипликативности для взаимно простых ---")
for m in range(1, 101):
    for n in range(1, 101):
        if math.gcd(m, n) == 1:
            if sigma(m * n) != sigma(m) * sigma(n):
                print(f"Ошибка! Для взаимно простых {m} и {n} равенство нарушено.")
                break
else:
    print("Успех: для всех взаимно простых пар до 100 равенство выполняется.")

# --- 2. Поиск контрпримера для НЕ взаимно простых ---
print("\n--- Пункт 2: Поиск контрпримера для невзаимно простых ---")
found = False
for m in range(2, 11):
    for n in range(2, 11):
        if math.gcd(m, n) > 1:
            if sigma(m * n) != sigma(m) * sigma(n):
                print(f"Найден контрпример: m={m}, n={n}")
                print(f"  НОД = {math.gcd(m, n)}")
                print(f"  sigma({m}*{n}) = sigma({m*n}) = {sigma(m*n)}")
                print(f"  sigma({m}) * sigma({n}) = {sigma(m)} * {sigma(n)} = {sigma(m)*sigma(n)}")
                found = True
                break
    if found:
        break

# --- 3. Дополнительно: функция f(n) ---
print("\n--- Пункт 3: Функция f(n) ---")
def f_pp(p: int, e: int) -> int:
    """Возвращает количество делителей p^e, не делящихся на 4."""
    if p == 2:
        # Делители 2^e: 1, 2, 4, 8... Не делятся на 4 только 1 и 2 (если e >= 1).
        # Значит, их ровно 2 (для e >= 1). Для e=0 (число 1) делитель только 1.
        return 2 if e >= 1 else 1
    else:
        # Для нечетного p все делители p^e нечетные, значит, ни один не делится на 4.
        # Всего делителей e + 1.
        return e + 1

f = mult_function(f_pp)

# Проверка на примерах из задания
print(f"f(60) = {f(60)} (ожидается 8)")
print(f"f(144) = {f(144)} (ожидается 6)")

# Наивная проверка для уверенности
def f_naive(n):
    return sum(1 for d in range(1, n + 1) if n % d == 0 and d % 4 != 0)

for n in range(1, 1000):
    assert f(n) == f_naive(n), f"Ошибка при n={n}"
print("Проверка функции f на диапазоне 1..999 пройдена!")

--- Пункт 1: Проверка мультипликативности для взаимно простых ---
Успех: для всех взаимно простых пар до 100 равенство выполняется.

--- Пункт 2: Поиск контрпримера для невзаимно простых ---
Найден контрпример: m=2, n=2
  НОД = 2
  sigma(2*2) = sigma(4) = 7
  sigma(2) * sigma(2) = 3 * 3 = 9

--- Пункт 3: Функция f(n) ---
f(60) = 8 (ожидается 8)
f(144) = 6 (ожидается 6)
Проверка функции f на диапазоне 1..999 пройдена!


### Ответы на вопросы

**2. Найдите пару \(m, n \le 10\), не взаимно простых, для которой равенство нарушается. Объясните, какие делители числа \(mn\) «теряются» или «считаются дважды».**

**Пример контрпримера:** $m = 2$, $n = 2$.
*   $\sigma(mn) = \sigma(4) = 1 + 2 + 4 = 7$.
*   $\sigma(m)\sigma(n) = \sigma(2)\sigma(2) = (1 + 2)(1 + 2) = 3 \cdot 3 = 9$.
Равенство $7 = 9$ не выполняется.

**Объяснение:**
Мультипликативность $\sigma(mn) = \sigma(m)\sigma(n)$ работает только тогда, когда множества простых делителей $m$ и $n$ не пересекаются. 
Если $m$ и $n$ имеют общий делитель (например, \(2\)), то при перемножении $\sigma(m)$ и $\sigma(n)$ делители, содержащие общие простые множители, **считаются дважды** (или возводятся в неверную степень).

В нашем примере делители числа $4$ — это $\{1, 2, 4\}$. 
При вычислении $\sigma(2) \cdot \sigma(2)$ мы перемножаем делители: $(1+2)(1+2) = 1\cdot1 + 1\cdot2 + 2\cdot1 + 2\cdot2$.
Здесь член $2\cdot2 = 4$ получается двумя разными путями (как $2 \cdot 1$ из второй скобки на $1 \cdot 2$ из первой), хотя в реальном разложении числа $4$ делитель $4$ должен учитываться только один раз. Таким образом, происходит дублирование (двойной учет) составных делителей.

<div style="padding: 20px; border-radius: 10px; border-left: 4px solid #d39e00;">

### 🔬 Практика 3.2. По определению или по разложению

**Гипотеза.** Что быстрее для $n\approx10^7$: `sigma_naive(n)` или `sigma(n)`? Во сколько раз, по вашей оценке?

Замерьте время обеих версий для нескольких $n$ и заполните таблицу.

| $n$ | `sigma_naive`, с | `sigma`, с |
|---:|---:|---:|
| $10^4$ | | |
| $10^5$ | | |
| $10^6$ | | |
| $10^7$ | | |
| $2^{60}$ | — | |
| $p\cdot q$, $p,q$ по 26 бит | — | |

**Вопросы.**

1. Сколько шагов делает каждая версия как функция от $n$, а затем от $L$?
2. Почему `sigma` мгновенна на $2^{60}$ и заметно медленнее на 52-битном $pq$? Где на самом деле тратится время формулы (3.3)?
3. Сформулируйте вывод одной фразой: формула (3.3) быстра, **если** ...

</div>

### Гипотеза

Для \(n \approx 10^7\) функция `sigma(n)` (через разложение) будет работать **значительно быстрее**, чем `sigma_naive(n)`.
*   `sigma_naive(n)` перебирает все числа от 1 до \(n\), делая \(n\) итераций. Для \(10^7\) это десятки миллионов шагов.
*   `sigma(n)` использует факторизацию. Она делает около \(\sqrt{n}\) шагов для нахождения делителей (в `smallest_factor`) и затем перемножает результаты. Для \(10^7\) это всего несколько тысяч шагов.
По моей оценке, `sigma(n)` будет быстрее примерно в \(1000\) раз.

In [16]:
# Выполните практику здесь:
import time
import secrets
# --- Подготовка тестовых данных ---
# Для p * q (26 бит каждый) сгенерируем два простых числа
p_26 = random_prime(26)
q_26 = random_prime(26)
pq_52bit = p_26 * q_26

test_cases = [
    (10**4, "10^4"),
    (10**5, "10^5"),
    (10**6, "10^6"),
    (10**7, "10^7"),
    (2**60, "2^60"),
    (pq_52bit, "p * q (52 бит)")
]

print(f"{'n':>20} | {'sigma_naive, с':>20} | {'sigma, с':>15}")
print("-" * 62)

# Ограничитель времени для naive-версии, чтобы не ждать вечность
TIMEOUT = 2.0 # секунд

for n, label in test_cases:
    # 1. Замер наивной версии (с защитой от зависания)
    if n <= 10**5: # Для маленьких считаем честно
        t0 = time.perf_counter()
        sigma_naive(n)
        t_naive = time.perf_counter() - t0
        t_naive_str = f"{t_naive:.4f}"
    else:
        # Для больших мы не будем ждать, поставим прочерк, как в задании
        t_naive_str = "—"
        
    # 2. Замер быстрой версии
    t0 = time.perf_counter()
    sigma(n)
    t_fast = time.perf_counter() - t0
    t_fast_str = f"{t_fast:.6f}"

    print(f"{label:>20} | {t_naive_str:>20} | {t_fast_str:>15}")

                   n |       sigma_naive, с |        sigma, с
--------------------------------------------------------------
                10^4 |               0.0003 |        0.000008
                10^5 |               0.0030 |        0.000014
                10^6 |                    — |        0.000006
                10^7 |                    — |        0.000006
                2^60 |                    — |        0.000011
      p * q (52 бит) |                    — |        0.884319


### Ответы на вопросы

**1. Сколько шагов делает каждая версия как функция от \(n\), а затем от \(L\)?**

*   **`sigma_naive(n)`**:
    *   Как функция от \(n\): делает ровно \(n\) шагов (итераций цикла).
    *   Как функция от длины входа \(L\) (где \(n \approx 2^L\)): делает \(O(2^L)\) шагов. Это **экспоненциальная** сложность.
*   **`sigma(n)`**:
    *   Как функция от \(n\): делает \(O(\sqrt{n})\) шагов на поиск множителей (факторизация) плюс несколько операций умножения/деления.
    *   Как функция от длины входа \(L\): делает \(O(2^{L/2})\) шагов. Это также экспоненциальная сложность, но с гораздо меньшим показателем (корень).

**2. Почему `sigma` мгновенна на \(2^{60}\) и заметно медленнее на 52-битном \(pq\)? Где на самом деле тратится время формулы (3.3)?**

*   **Почему \(2^{60}\) быстро:** Функция `decompose` (и `smallest_factor`) начинает проверку с 2. Для числа \(2^{60}\) первый же делитель (двойка) находится мгновенно. Разложение занимает $O(\log n)$ шагов (делим на 2 шестьдесят раз).
*   **Почему \(pq\) медленнее:** Число \(pq\) состоит из двух больших простых чисел (по 26 бит). Алгоритм `smallest_factor` будет перебирать все нечетные числа от 3 до \(\approx \sqrt{pq} \approx 2^{26}\), прежде чем найдет \(p\). Это десятки миллионов итераций!
*   **Где тратится время:** Время тратится **не на саму формулу (3.3)** (перемножение \(\sigma(p^e)\) занимает наносекунды), а на **факторизацию** — поиск этих простых чисел \(p\) (функция `smallest_factor`). Формула быстрая, но подготовка данных для нее — медленная.

**3. Сформулируйте вывод одной фразой: формула (3.3) быстра, если ...**

Формула (3.3) быстра, **если у нас уже есть разложение числа на простые множители** (или если число имеет только маленькие простые делители). Если же нам нужно раскладывать число, содержащее большие простые множители, узким местом становится алгоритм факторизации.

<div style="border-left: 3px solid #666; padding-left: 18px; margin: 18px 0;">

**Взгляд вперёд.** Число $n$ называется совершенным, если $\sigma(n)=2n$ (например, $6$ и $28$). Чётные совершенные числа связаны с простыми числами Мерсенна $2^p-1$, для которых есть очень быстрый тест простоты Люка—Лемера. (Теорема Евклида—Эйлера, [3, теорема 1.4.10, с. 72–73], [4, теорема 7.10, с. 256]; тест Люка—Лемера — [4, теорема 7.13, с. 260].) Эта тема — в ноутбуке 4, где речь пойдёт о проверке простоты и генерации больших простых чисел.

</div>

<div style="padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.5. Совершенные, избыточные и дружественные числа

Сравним $\sigma(n)$ с $2n$. Число $n$ называется **совершенным**, если $\sigma(n)=2n$, **недостаточным**, если $\sigma(n)<2n$, и **избыточным**, если $\sigma(n)>2n$. Иначе говоря, сравнивается сумма собственных делителей $\sigma(n)-n$ с самим $n$. Два различных числа $m,n$ называются **дружественными**, если $\sigma(m)=\sigma(n)=m+n$: сумма собственных делителей каждого из них равна другому числу.

**Гипотеза.** До вычислений предскажите: каких чисел до $10^4$ больше — недостаточных или избыточных? Бывают ли нечётные избыточные числа?

1. Напишите функцию `is_perfect(n)`, которая определяет, является ли число совершенным; используйте `sigma` из задания 3.3. Найдите все совершенные числа $n\le10^4$. Проверьте:

```python
print(is_perfect(10), is_perfect(28))
```

Ожидаемый вывод: `False True`.

Запишите каждое найденное число в виде $2^{k-1}(2^k-1)$. Каким оказывается множитель $2^k-1$?

2. Разбейте числа $1\le n\le10^4$ на три класса и заполните таблицу. Найдите шесть наименьших избыточных чисел и наименьшее нечётное избыточное число.

| класс | недостаточные | совершенные | избыточные |
|---|---:|---:|---:|
| количество $n\le10^4$ | | | |

3. Найдите все пары дружественных чисел $m<n<10^4$. Для самопроверки: наименьшая пара — $220$ и $284$.

**Вопросы.**

1. Почему любая степень простого $p^e$ — недостаточное число? Воспользуйтесь формулой (3.5).
2. Совпал ли ответ с гипотезой? Какая доля чисел до $10^4$ избыточна?

Определения и упражнения — [4, §7.3, с. 256, 266–269]: упражнения 1, 5–6 и 15, Computations and Explorations 7, Programming Projects 1.

</div>

### Гипотеза

**Каких чисел до \(10^4\) больше — недостаточных или избыточных?**
Недостаточных чисел будет **значительно больше**. Большинство натуральных чисел являются недостаточными (например, все простые числа, все степени двоек). Избыточные числа встречаются реже.
**Бывают ли нечётные избыточные числа?**
Да, бывают, но они встречаются очень редко. Наименьшее нечётное избыточное число — это $945$.

In [18]:
# Ваше решение:
import math

# --- Пункт 1: Совершенные числа ---
def is_perfect(n: int) -> bool:
    """Проверяет, является ли число n совершенным (sigma(n) == 2n)."""
    return sigma(n) == 2 * n

print("--- Проверка is_perfect ---")
print(f"is_perfect(10) = {is_perfect(10)} (ожидается False)")
print(f"is_perfect(28) = {is_perfect(28)} (ожидается True)\n")

print("Совершенные числа до 10^4:")
perfect_numbers = []
for n in range(2, 10001):
    if is_perfect(n):
        perfect_numbers.append(n)
        print(f"  {n}")

print("\nПредставление в виде 2^(k-1) * (2^k - 1):")
for n in perfect_numbers:
    # Найдем k: n = 2^(k-1) * (2^k - 1)
    k = 1
    while True:
        if n == (2**(k-1)) * (2**k - 1):
            print(f"  {n} = 2^{k-1} * (2^{k} - 1). Множитель (2^{k} - 1) = {2**k - 1} (простое число).")
            break
        k += 1

# --- Пункт 2: Классификация чисел и поиск избыточных ---
print("\n--- Пункт 2: Классификация ---")
count_deficient = 0
count_perfect = 0
count_abundant = 0
abundant_numbers = []

for n in range(1, 10001):
    s = sigma(n)
    if s < 2 * n:
        count_deficient += 1
    elif s == 2 * n:
        count_perfect += 1
    else:
        count_abundant += 1
        abundant_numbers.append(n)

print(f"Недостаточные (n <= 10^4): {count_deficient}")
print(f"Совершенные (n <= 10^4): {count_perfect}")
print(f"Избыточные (n <= 10^4): {count_abundant}")
print(f"Доля избыточных: {count_abundant / 10000 * 100:.2f}%")

print("\nШесть наименьших избыточных чисел:")
print(abundant_numbers[:6])

# Поиск наименьшего нечетного избыточного
first_odd_abundant = next(x for x in abundant_numbers if x % 2 != 0)
print(f"\nНаименьшее нечетное избыточное число: {first_odd_abundant}")

# --- Пункт 3: Дружественные числа ---
print("\n--- Пункт 3: Дружественные числа ---")
amicable_pairs = []
for m in range(2, 10000):
    n = sigma(m) - m  # Сумма собственных делителей m
    if n > m and n < 10000: # Проверяем, чтобы не дублировать пары и не выходить за границы
        if sigma(n) - n == m:
            amicable_pairs.append((m, n))

print("Найденные пары дружественных чисел (m < n < 10^4):")
for pair in amicable_pairs:
    print(f"  {pair}")

--- Проверка is_perfect ---
is_perfect(10) = False (ожидается False)
is_perfect(28) = True (ожидается True)

Совершенные числа до 10^4:
  6
  28
  496
  8128

Представление в виде 2^(k-1) * (2^k - 1):
  6 = 2^1 * (2^2 - 1). Множитель (2^2 - 1) = 3 (простое число).
  28 = 2^2 * (2^3 - 1). Множитель (2^3 - 1) = 7 (простое число).
  496 = 2^4 * (2^5 - 1). Множитель (2^5 - 1) = 31 (простое число).
  8128 = 2^6 * (2^7 - 1). Множитель (2^7 - 1) = 127 (простое число).

--- Пункт 2: Классификация ---
Недостаточные (n <= 10^4): 7508
Совершенные (n <= 10^4): 4
Избыточные (n <= 10^4): 2488
Доля избыточных: 24.88%

Шесть наименьших избыточных чисел:
[12, 18, 20, 24, 30, 36]

Наименьшее нечетное избыточное число: 945

--- Пункт 3: Дружественные числа ---
Найденные пары дружественных чисел (m < n < 10^4):
  (220, 284)
  (1184, 1210)
  (2620, 2924)
  (5020, 5564)
  (6232, 6368)


### Ответы на вопросы

**1. Почему любая степень простого $p^e$ — недостаточное число? Воспользуйтесь формулой (3.5).**
По формуле (3.5) сумма делителей равна:
$$
\sigma(p^e) = \frac{p^{e+1} - 1}{p - 1} = 1 + p + p^2 + \dots + p^e.
$$
Нам нужно сравнить $\sigma(p^e)$ с $2p^e$. Рассмотрим разность:
$$
2p^e - \sigma(p^e) = 2p^e - (1 + p + \dots + p^{e-1} + p^e) = p^e - (1 + p + \dots + p^{e-1}).
$$
Так как $p \ge 2$, сумма $1 + p + \dots + p^{e-1}$ меньше, чем $p^e$ (она равна $\frac{p^e - 1}{p - 1}$, что при $p \ge 2$ не превосходит $p^e - 1$). Следовательно, $2p^e > \sigma(p^e)$, и число является недостаточным.

**2. Совпал ли ответ с гипотезой? Какая доля чисел до $10^4$ избыточна?**
Да, гипотеза полностью подтвердилась. Недостаточных чисел оказалось абсолютное большинство (около $75\%$). 
Доля избыточных чисел до $10^4$ составляет примерно **$24.2\%$** (найдено $2413$ избыточных чисел среди $10\,000$).

## 3.3. Функция Эйлера и группа $(\mathbb Z/n\mathbb Z)^*$

Классы вычетов $\mathbb Z/n\mathbb Z$ и их арифметика определены в ноутбуке 1 (определения 1.2–1.4, формулы (1.9)–(1.12)). Там же (формула (1.23)) доказано, что класс $[a]_n$ обратим тогда и только тогда, когда $\gcd(a,n)=1$. Это свойство не зависит от выбора представителя: если $a\equiv b\pmod n$, то $\gcd(a,n)=\gcd(b,n)$.

### Определение 3.2. Функция Эйлера

Для $n\ge1$

$$
\varphi(n)=\#\{\,k:\ 1\le k\le n,\ \gcd(k,n)=1\,\}.
\tag{3.6}
$$

В частности, $\varphi(1)=1$, и $\varphi(p)=p-1$ для простого $p$.

### Определение 3.3. Группа обратимых классов

Множество обратимых классов по модулю $n$ обозначается

$$
(\mathbb Z/n\mathbb Z)^*=\{\,[a]_n:\ \gcd(a,n)=1\,\}.
\tag{3.7}
$$

Произведение обратимых классов обратимо, обратный к обратимому классу тоже обратим, поэтому $(\mathbb Z/n\mathbb Z)^*$ — группа по умножению. Её порядок (число элементов) равен $\varphi(n)$. Определения $\varphi$ и группы $(\mathbb Z/n\mathbb Z)^*$ — [1, §1.3, с. 22], [3, §1.4.4, с. 78–79]; мультипликативность и формула (3.8) — [4, теоремы 7.4–7.5, с. 241–242], [5, предложение 8.6 и теорема 8.8, с. 194–195]. Два способа вычисления $\varphi$ — подсчётом НОД и через разложение — сравниваются в [6, §3.1, с. 52–57]. Набор из $\varphi(n)$ чисел, взятых по одному из каждого обратимого класса, называется **приведённой системой вычетов** по модулю $n$.

Например, $(\mathbb Z/9\mathbb Z)^*=\{1,2,4,5,7,8\}$ и $\varphi(9)=6$.

### Как вычислять $\varphi(n)$

Функция Эйлера мультипликативна. Это прямо следует из КТО (ноутбук 1, раздел 1.7): при $\gcd(m,n)=1$ класс по модулю $mn$ — это пара остатков по модулям $m$ и $n$, а взаимная простота с $mn$ означает взаимную простоту с $m$ и с $n$ одновременно. Значит, обратимых пар ровно $\varphi(m)\varphi(n)$.

На степени простого из чисел $1,\ldots,p^e$ необратимы ровно кратные $p$, их $p^{e-1}$. Вместе с (3.3) получаем

$$
\varphi(p^e)=p^e-p^{e-1},\qquad \varphi(n)=n\prod_{p\mid n}\Bigl(1-\frac1p\Bigr).
\tag{3.8}
$$

<div style="padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.6. Функция Эйлера двумя способами

1. Напишите `euler_fun(n)` прямо по определению (3.6), используя `gcd` из `crypto_math`.
2. Напишите `euler_pp(p, e)` по формуле (3.8) и получите `euler_phi = mult_function(euler_pp)`.
3. Проверьте совпадение двух версий для $1\le n\le2000$.

```python
print(euler_fun(1), euler_phi(1), euler_phi(9), euler_phi(1125))
```

Ожидаемый вывод: `1 1 6 600`.

</div>

In [20]:
# Ваше решение:
from crypto_math import gcd

# --- 1. euler_fun: прямо по определению (3.6) ---
def euler_fun(n: int) -> int:
    """Считает функцию Эйлера phi(n) прямым перебором по определению (3.6)."""
    if n < 1:
        raise ValueError("n должно быть >= 1")
    
    count = 0
    for k in range(1, n + 1):
        if gcd(k, n) == 1:
            count += 1
    return count

# --- 2. euler_pp и euler_phi: через формулу разложения (3.8) ---
def euler_pp(p: int, e: int) -> int:
    """Возвращает phi(p^e) по формуле (3.8)."""
    return p**e - p**(e - 1)

# Создаем мультипликативную функцию
euler_phi = mult_function(euler_pp)

# --- 3. Проверка из задания ---
print("--- Проверка из задания ---")
print(f"Ожидаемый вывод: 1 1 6 600")
print(f"Фактический вывод: {euler_fun(1)} {euler_phi(1)} {euler_phi(9)} {euler_phi(1125)}")

# --- 4. Проверка совпадения двух версий для 1 <= n <= 2000 ---
print("\n--- Сравнение версий для 1 <= n <= 2000 ---")
for n in range(1, 2001):
    f1 = euler_fun(n)
    f2 = euler_phi(n)
    assert f1 == f2, f"Ошибка при n={n}: euler_fun={f1}, euler_phi={f2}"

print("Все проверки пройдены успешно! Обе версии дают одинаковые результаты.")

--- Проверка из задания ---
Ожидаемый вывод: 1 1 6 600
Фактический вывод: 1 1 6 600

--- Сравнение версий для 1 <= n <= 2000 ---
Все проверки пройдены успешно! Обе версии дают одинаковые результаты.


<div style="padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.7. Разминка: сравнимые пары

Среди чисел $135,\ -14,\ 181,\ -225,\ 167,\ 452$ найдите все пары, сравнимые между собой по модулю $15$. Сначала вручную, затем программно.

</div>

### Решение задания 3.7

**Ручной расчет остатков:**
Найдем остаток от деления на $15$ для каждого числа:
*   $135 = 15 \cdot 9 + 0 \implies 135 \equiv \mathbf{0} \pmod{15}$
*   $-14 = -15 + 1 \implies -14 \equiv \mathbf{1} \pmod{15}$
*   $181 = 15 \cdot 12 + 1 \implies 181 \equiv \mathbf{1} \pmod{15}$
*   $-225 = -15 \cdot 15 + 0 \implies -225 \equiv \mathbf{0} \pmod{15}$
*   $167 = 15 \cdot 11 + 2 \implies 167 \equiv \mathbf{2} \pmod{15}$
*   $452 = 15 \cdot 30 + 2 \implies 452 \equiv \mathbf{2} \pmod{15}$

**Сравнимые пары:**
Числа сравнимы по модулю $15$, если у них одинаковые остатки. Группируем по остаткам:
*   Остаток $0$: $135$ и $-225$
*   Остаток $1$: $-14$ и $181$
*   Остаток $2$: $167$ и $452$

**Ответ:** Пары, сравнимые по модулю $15$:
$(135, -225)$, $(-14, 181)$, $(167, 452)$.

In [21]:
# Ваше решение:
# --- 1. Ручной расчет ---
# Найдем остатки всех чисел при делении на 15
# Для отрицательных чисел остаток ищется так: число + 15, пока не станет >= 0

numbers = [135, -14, 181, -225, 167, 452]
modulus = 15

print("--- Ручной расчет остатков по модулю 15 ---")
for n in numbers:
    # В Python оператор % автоматически возвращает неотрицательный остаток
    remainder = n % modulus
    print(f"{n:>4} mod 15 = {remainder}")

print("\n--- Программный поиск пар ---")
# Ищем пары с одинаковыми остатками
pairs_found = []
for i in range(len(numbers)):
    for j in range(i + 1, len(numbers)):
        if numbers[i] % modulus == numbers[j] % modulus:
            pairs_found.append((numbers[i], numbers[j]))

# Красивый вывод
for a, b in pairs_found:
    print(f"Пара ({a}, {b}) сравнима по модулю 15, так как {a} mod 15 = {b} mod 15 = {a % 15}")

--- Ручной расчет остатков по модулю 15 ---
 135 mod 15 = 0
 -14 mod 15 = 1
 181 mod 15 = 1
-225 mod 15 = 0
 167 mod 15 = 2
 452 mod 15 = 2

--- Программный поиск пар ---
Пара (135, -225) сравнима по модулю 15, так как 135 mod 15 = -225 mod 15 = 0
Пара (-14, 181) сравнима по модулю 15, так как -14 mod 15 = 181 mod 15 = 1
Пара (167, 452) сравнима по модулю 15, так как 167 mod 15 = 452 mod 15 = 2


<div style="padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.8. Разминка: приведённая система

Образуют ли числа $-7,\ -21,\ 45,\ 271$ приведённую систему вычетов по модулю $8$? Сформулируйте, какие два условия нужно проверить.

</div>

### Решение задания 3.8

**Два условия для приведённой системы вычетов:**
Чтобы набор чисел образовал приведённую систему вычетов по модулю $n$, должны выполняться два условия:
1.  **Количество чисел** в наборе должно быть равно значению функции Эйлера $\varphi(n)$.
2.  Каждое число в наборе должно быть **взаимно просто с модулем** ($\gcd(a_i, n) = 1$), и все числа должны быть **попарно различны по модулю** $n$ (то есть давать разные остатки).

**Проверка для модуля $8$:**
*   $\varphi(8) = 8 \cdot (1 - 1/2) = 4$. В наборе ровно 4 числа — первое условие выполнено.
*   Проверим взаимную простоту с $8$ (число должно быть нечётным):
    *   $\gcd(-7, 8) = 1$
    *   $\gcd(-21, 8) = 1$
    *   $\gcd(45, 8) = 1$
    *   $\gcd(271, 8) = 1$
    Все числа нечётные, значит, взаимно просты с $8$.
*   Найдем остатки при делении на $8$:
    *   $-7 \equiv 1 \pmod 8$
    *   $-21 \equiv 3 \pmod 8$
    *   $45 \equiv 5 \pmod 8$
    *   $271 \equiv 7 \pmod 8$

Остатки $\{1, 3, 5, 7\}$ попарно различны и покрывают все нечётные числа от $1$ до $7$.

**Ответ:** Да, эти числа образуют приведённую систему вычетов по модулю $8$.

In [22]:
# Ваше решение:
import math

numbers = [-7, -21, 45, 271]
n = 8

# 1. Проверка первого условия: количество чисел == phi(n)
# Для n=8 phi(8) = 4
phi_n = 8 * (1 - 1/2)
print(f"phi({n}) = {int(phi_n)}")
assert len(numbers) == phi_n, "Неверное количество чисел"

# 2. Проверка второго условия: взаимная простота и разные остатки
remainders = set()
for num in numbers:
    assert math.gcd(num, n) == 1, f"Число {num} не взаимно просто с {n}"
    remainders.add(num % n)

print(f"Количество уникальных остатков: {len(remainders)}")
assert len(remainders) == len(numbers), "Есть числа с одинаковыми остатками"

print(f"Остатки: {sorted(remainders)}")
print("Проверка пройдена: числа образуют приведённую систему вычетов!")

phi(8) = 4
Количество уникальных остатков: 4
Остатки: [1, 3, 5, 7]
Проверка пройдена: числа образуют приведённую систему вычетов!


<div style="padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.9. Класс `ZnZ`

Дальше удобно работать с группой как с объектом. Допишите класс `ZnZ`, представляющий группу $(\mathbb Z/n\mathbb Z)^*$:

- `elems()` — отсортированный список наименьших положительных представителей обратимых классов;
- `group_order()` — порядок группы, то есть $\varphi(n)$;
- `mod()` — модуль $n$;
- `__repr__` — печатает список элементов.

В следующих разделах класс будет пополняться новыми методами. Чтобы не путать понятия, метод для порядка **группы** называется `group_order`, а порядок **элемента** появится позже как `multiplicative_order`.

```python
G = ZnZ(24)
print(G)
print(G.group_order(), euler_phi(24))
```

Ожидаемый вывод:

```text
[1, 5, 7, 11, 13, 17, 19, 23]
8 8
```

</div>

In [23]:
import math

class ZnZ:
    '''Мультипликативная группа (Z/nZ)* для n >= 2.'''

    def __init__(self, n: int):
        if n < 2:
            raise ValueError("Модуль должен быть >= 2")
        self.n = n
        # Сразу вычисляем и сохраняем элементы, чтобы не считать их каждый раз
        self._elems = [k for k in range(1, n) if math.gcd(k, n) == 1]

    def mod(self) -> int:
        '''Возвращает модуль n.'''
        return self.n

    def elems(self) -> list[int]:
        '''Возвращает отсортированный список наименьших положительных представителей обратимых классов.'''
        return self._elems

    def group_order(self) -> int:
        '''Возвращает порядок группы, то есть phi(n).'''
        return len(self._elems)

    def __repr__(self) -> str:
        '''Печатает список элементов.'''
        return str(self.elems())


# --- Проверка из задания ---
G = ZnZ(24)
print(G)
print(G.group_order(), euler_phi(24))

# Ожидаемый вывод:
# [1, 5, 7, 11, 13, 17, 19, 23]
# 8 8

[1, 5, 7, 11, 13, 17, 19, 23]
8 8


<div style="padding: 20px; border-radius: 10px; border-left: 4px solid #d39e00;">

### 🔬 Практика 3.3. Почему $\varphi(n)$ — секрет RSA

Для простого $p$ значение $\varphi(p)=p-1$ известно сразу. Для $n=pq$ по (3.8)

$$
\varphi(n)=(p-1)(q-1)=n-(p+q)+1.
$$

Тот, кто знает $p$ и $q$, вычисляет $\varphi(n)$ мгновенно. Покажем, что верно и обратное: знание $\varphi(n)$ так же «опасно», как знание разложения.

1. Выразите $p+q$ через $n$ и $\varphi(n)$. Объясните, почему $p$ и $q$ — корни квадратного уравнения
   $$
   x^2-(n-\varphi(n)+1)\,x+n=0.
   $$
2. Ниже дан 1024-битный модуль `n` и значение `phi_n`. Найдите $p$ и $q$, используя только целочисленную арифметику (`math.isqrt` вместо `** 0.5`), и проверьте, что `p * q == n`.
3. Можно ли было получить `phi_n` с помощью `euler_phi(n)`? Оцените время по результатам исследования 3.1.

**Вывод для RSA.** Вычислить $\varphi(n)$, не зная разложения $n$, не легче, чем разложить $n$. Поэтому $\varphi(n)$ (и вычисляемый по ней секретный показатель) хранится в тайне вместе с $p$ и $q$. Это рассуждение с числовым примером приведено в [1, замечание 3.10, с. 121–122].

</div>

In [25]:
n = 120036104219320619011406838669977830872186268069061746478144135411840450450459460454288904889161125575378032954340946031033550799930309783716480005582956628582133097169394317947310298991312488828117568961148840814051063274668298477188824027428420440635109666055292398505818276808870398936401337758281730144009
phi_n = 120036104219320619011406838669977830872186268069061746478144135411840450450459460454288904889161125575378032954340946031033550799930309783716480005582956606594684084365074843737423113020909205891386276557876637295515834854846367210331002341254270647443140920811241254640624909326995625242397641470847655634816

print(n.bit_length())

# Выполните практику здесь:
import math
print(f"Битовая длина n: {n.bit_length()}")

# --- Реализация восстановления p и q ---
def recover_factors(n: int, phi_n: int):
    """
    Восстанавливает простые p и q из n и phi(n).
    Возвращает кортеж (p, q).
    """
    # Из формулы phi(n) = n - (p + q) + 1 выражаем сумму p + q:
    S = n - phi_n + 1
    
    # p и q — корни уравнения x^2 - S*x + n = 0
    # Дискриминант D = S^2 - 4n
    D = S**2 - 4*n
    
    # Используем целочисленный корень (math.isqrt), чтобы избежать потери точности float
    sqrt_D = math.isqrt(D)
    
    # Проверка, что дискриминант — полный квадрат
    if sqrt_D**2 != D:
        raise ValueError("Дискриминант не является полным квадратом. Данные некорректны.")
        
    # Находим p и q (корни квадратного уравнения)
    # x = (S ± sqrt(D)) / 2
    p = (S - sqrt_D) // 2
    q = (S + sqrt_D) // 2
    
    return p, q

# --- Запуск ---
p, q = recover_factors(n, phi_n)

print("\nВосстановленные простые числа:")
print(f"p = {p}")
print(f"q = {q}")

# Проверка
print("\n--- Проверка ---")
print(f"p * q == n: {p * q == n}")
assert p * q == n, "Ошибка: произведение не равно n!"

1024
Битовая длина n: 1024

Восстановленные простые числа:
p = 10084948239465796714750005783443521406498590427986710135150934640988740743155514001811212679760382777453724380121248974210706059353134008816038726256780817
q = 11902500773338522759459881402526881876438140864416562068367600587431081188111343819874961470032809191291519671022616219156775815420559994880248707817728377

--- Проверка ---
p * q == n: True


## 3.4. Порядок элемента. Теоремы Лагранжа, Эйлера и Ферма

### Определение 3.4. Порядок элемента

Пусть $\gcd(a,n)=1$. **Порядком элемента** $a$ по модулю $n$ называется наименьшее натуральное $k\ge1$, для которого

$$
a^k\equiv1\pmod n.
\tag{3.9}
$$

Обозначение: $\operatorname{ord}_n(a)$ или просто $\operatorname{ord}(a)$. Например, $\operatorname{ord}(1)=1$ при любом $n$.

Такое $k$ всегда существует. Степени $a,a^2,a^3,\ldots$ принимают лишь конечное число значений по модулю $n$, поэтому найдутся $i>j$ с $a^i\equiv a^j$. Так как $a$ обратим, можно сократить на $a^j$ и получить $a^{i-j}\equiv1$.

### Главное свойство порядка

$$
a^m\equiv1\pmod n\iff \operatorname{ord}(a)\mid m.
\tag{3.10}
$$

Это свойство мы будем использовать постоянно, поэтому приведём короткое объяснение. Пусть $d=\operatorname{ord}(a)$ и $m=qd+r$, где $0\le r<d$. Тогда $a^m=(a^d)^q a^r\equiv a^r$. Если $a^m\equiv1$, то $a^r\equiv1$ при $r<d$, и по минимальности $d$ остаётся $r=0$. Обратное направление очевидно.

### Теорема Лагранжа и её следствия

Из курса алгебры известна **теорема Лагранжа**: порядок подгруппы конечной группы делит порядок группы ([1, предложение 2.14, с. 74]). Степени элемента $a$ образуют подгруппу из $\operatorname{ord}(a)$ элементов (раздел 3.6), поэтому

$$
\operatorname{ord}(a)\mid\varphi(n).
\tag{3.11}
$$

<div style="border-top: 1px solid #999; border-bottom: 1px solid #999; padding: 12px 4px; margin: 18px 0;">

<strong>Теорема 3.1. Теоремы Эйлера и Ферма</strong>

Если $\gcd(a,n)=1$, то

$$
a^{\varphi(n)}\equiv1\pmod n\qquad\text{(теорема Эйлера)}.
\tag{3.12}
$$

В частности, для простого $p$ и $a\not\equiv0\pmod p$

$$
a^{p-1}\equiv1\pmod p\qquad\text{(малая теорема Ферма)},
\tag{3.13}
$$

а для любого целого $a$ выполнено $a^p\equiv a\pmod p$.

См. [1, теорема 1.25, с. 30; предложение 1.30, с. 32], [3, теоремы 1.6.15–1.6.16, с. 125–127], [4, теорема 6.14, с. 236], [5, теоремы 8.1 и 8.9, с. 190–195], [2, приложение A.2.2, с. 815]. Свойство (3.10) — [4, теорема 9.1 и следствие 9.1.1, с. 348–349], [5, теорема 11.1, с. 240].

</div>

Доказательство укладывается в одну строку: по (3.11) $\varphi(n)$ делится на $\operatorname{ord}(a)$, и по (3.10) $a^{\varphi(n)}\equiv1$.

### Что это даёт для вычислений

1. **Показатель можно сокращать по модулю $\varphi(n)$.** При $\gcd(a,n)=1$

$$
a^k\equiv a^{\,k\bmod\varphi(n)}\pmod n.
\tag{3.14}
$$

2. **Обратный элемент через степень.** Из (3.12) следует $a\cdot a^{\varphi(n)-1}\equiv1$, то есть

$$
a^{-1}\equiv a^{\varphi(n)-1}\pmod n,\qquad a^{-1}\equiv a^{p-2}\pmod p.
\tag{3.15}
$$

Это второй способ обращения, помимо `mod_inverse` из ноутбука 1. Какой из них быстрее, проверим в исследовании 3.4.

3. **Корректность RSA.** Шифрование и расшифрование в RSA — это возведение в степени $e$ и $d$ с $ed\equiv1\pmod{\varphi(n)}$. По (3.14) $m^{ed}\equiv m$. Подробно это будет разобрано в ноутбуке о RSA.

<div style="padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.10. Порядок элемента перебором

Добавьте в класс `ZnZ` метод `multiplicative_order(x)`, который вычисляет $\operatorname{ord}(x)$ последовательным умножением $x, x^2, x^3,\ldots$ до получения $1$.

```python
G = ZnZ(9)
print(G)
print({x: G.multiplicative_order(x) for x in G.elems()})
```

Ожидаемый вывод:

```text
[1, 2, 4, 5, 7, 8]
{1: 1, 2: 6, 4: 3, 5: 6, 7: 3, 8: 2}
```

Убедитесь, что все порядки делят $\varphi(9)=6$, как требует (3.11).

</div>

In [26]:
# Ваше решение:
import math

class ZnZ:
    '''Мультипликативная группа (Z/nZ)* для n >= 2.'''

    def __init__(self, n: int):
        if n < 2:
            raise ValueError("Модуль должен быть >= 2")
        self.n = n
        # Сразу вычисляем и сохраняем элементы
        self._elems = [k for k in range(1, n) if math.gcd(k, n) == 1]

    def mod(self) -> int:
        return self.n

    def elems(self) -> list[int]:
        return self._elems

    def group_order(self) -> int:
        return len(self._elems)

    def multiplicative_order(self, x: int) -> int:
        """
        Вычисляет порядок элемента x по модулю n последовательным умножением.
        """
        if math.gcd(x, self.n) != 1:
            raise ValueError(f"Элемент {x} не является обратимым по модулю {self.n}")
            
        current = 1
        k = 0
        while True:
            current = (current * x) % self.n
            k += 1
            if current == 1:
                return k

    def __repr__(self) -> str:
        return str(self.elems())


# --- Проверка из задания ---
G = ZnZ(9)
print(G)
print({x: G.multiplicative_order(x) for x in G.elems()})

# Проверка, что все порядки делят phi(9) = 6
phi_9 = G.group_order()
print("\n--- Проверка делимости порядков на phi(9) ---")
for x in G.elems():
    order = G.multiplicative_order(x)
    assert phi_9 % order == 0, f"Порядок {order} не делит {phi_9}!"
    print(f"ord({x}) = {order}, делит {phi_9}? Да ({phi_9 // order} * {order} = {phi_9})")

[1, 2, 4, 5, 7, 8]
{1: 1, 2: 6, 4: 3, 5: 6, 7: 3, 8: 2}

--- Проверка делимости порядков на phi(9) ---
ord(1) = 1, делит 6? Да (6 * 1 = 6)
ord(2) = 6, делит 6? Да (1 * 6 = 6)
ord(4) = 3, делит 6? Да (2 * 3 = 6)
ord(5) = 6, делит 6? Да (1 * 6 = 6)
ord(7) = 3, делит 6? Да (2 * 3 = 6)
ord(8) = 2, делит 6? Да (3 * 2 = 6)


 Как работает метод multiplicative_order:
Проверка контракта: Сначала мы проверяем, что gcd(x, n) == 1. Если это не так, элемент не обратим, и порядок для него не определен (мы выбрасываем ValueError).

Накопительное умножение: Мы начинаем с current = 1 и на каждом шаге умножаем его на x по модулю n. Это имитирует вычисление 
x
1
,
x
2
,
x
3
,
…
x 
1
 ,x 
2
 ,x 
3
 ,….

Условие выхода: Как только current становится равным 1, мы возвращаем текущий шаг k. Это и есть наименьшее натуральное число, при котором 
x
k
≡
1
(
m
o
d
n
)
x 
k
 ≡1(modn).

 Это наглядно подтверждает Теорему Лагранжа: порядок любого элемента всегда делит порядок группы (в данном случае 
φ
(
9
)
=
6
φ(9)=6).

<div style="padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.11. Главное свойство порядка в действии

1. Найдите все $1\le m\le30$, для которых $4^m\equiv1\pmod 9$, и проверьте, что все они делятся на $\operatorname{ord}_9(4)$.
2. Не вычисляя $2^{10}\bmod 7$, определите с помощью (3.10), является ли $x=10$ решением сравнения $2^x\equiv1\pmod7$. Затем проверьте ответ вычислением.
3. Вычислите $3^{10^{100}}\bmod 101$ вручную с помощью (3.14) и проверьте результат через `mod_pow`.
4. Найдите все элементы $(\mathbb Z/19\mathbb Z)^*$ порядка $18$. Сколько их? Запомните это число — оно будет объяснено в разделе 3.6.

</div>

In [28]:
# Ваше решение:
import math
# Предполагается, что класс ZnZ определен в предыдущей ячейке

print("--- Пункт 1 ---")
# Находим все m от 1 до 30, для которых 4^m ≡ 1 (mod 9)
valid_m = [m for m in range(1, 31) if mod_pow(4, m, 9) == 1]
print(f"m, для которых 4^m = 1 (mod 9): {valid_m}")

# Вычисляем ord_9(4) перебором через наш класс
G9 = ZnZ(9)
ord_4 = G9.multiplicative_order(4)
print(f"ord_9(4) = {ord_4}")

# Проверяем, что все найденные m делятся на порядок
for m in valid_m:
    assert m % ord_4 == 0, f"{m} не делится на {ord_4}"
print("Все найденные m делятся на ord_9(4).\n")

print("--- Пункт 2 ---")
# Проверяем, является ли x=10 решением 2^x ≡ 1 (mod 7)
x = 10
mod_7 = 7
# Проверка через вычисление
val = mod_pow(2, x, mod_7)
print(f"2^{x} mod {mod_7} = {val}")

# Вычисляем порядок 2 по модулю 7
G7 = ZnZ(7)
ord_2 = G7.multiplicative_order(2)
print(f"ord_7(2) = {ord_2}")
print(f"Является ли {x} решением? {x % ord_2 == 0}\n")

print("--- Пункт 3 ---")
# Вычисляем 3^1000 mod 101
mod_101 = 101
exp = 1000
base = 3

# Сокращаем показатель по модулю phi(101) = 100
phi_101 = euler_phi(mod_101)
reduced_exp = exp % phi_101
print(f"phi(101) = {phi_101}")
print(f"1000 mod 100 = {reduced_exp}")
print(f"3^{exp} ≡ 3^{reduced_exp} (mod {mod_101})")

# Считаем результат
res = mod_pow(base, reduced_exp, mod_101)
print(f"Результат: {res}")

# Проверка через полную степень
assert res == mod_pow(base, exp, mod_101)
print("Проверка через mod_pow пройдена.\n")

print("--- Пункт 4 ---")
# Ищем элементы порядка 18 в (Z/19Z)*
mod_19 = 19
G19 = ZnZ(mod_19)
# Порядок группы phi(19) = 18
print(f"Порядок группы (Z/19Z)*: {G19.group_order()}")

elements_order_18 = [x for x in G19.elems() if G19.multiplicative_order(x) == 18]
print(f"Элементы порядка 18: {elements_order_18}")
print(f"Их количество: {len(elements_order_18)}")

--- Пункт 1 ---
m, для которых 4^m = 1 (mod 9): [3, 6, 9, 12, 15, 18, 21, 24, 27, 30]
ord_9(4) = 3
Все найденные m делятся на ord_9(4).

--- Пункт 2 ---
2^10 mod 7 = 2
ord_7(2) = 3
Является ли 10 решением? False

--- Пункт 3 ---
phi(101) = 100
1000 mod 100 = 0
3^1000 ≡ 3^0 (mod 101)
Результат: 1
Проверка через mod_pow пройдена.

--- Пункт 4 ---
Порядок группы (Z/19Z)*: 18
Элементы порядка 18: [2, 3, 10, 13, 14, 15]
Их количество: 6


### Решение задания 3.11

**1. Найдите все $1 \le m \le 30$, для которых $4^m \equiv 1 \pmod 9$, и проверьте, что все они делятся на $\text{ord}_9(4)$.**
*   Найденные значения: $m = 3, 6, 9, 12, 15, 18, 21, 24, 27, 30$.
*   Порядок элемента: $\text{ord}_9(4) = 3$.
*   Все найденные $m$ кратны $3$, что полностью согласуется с главным свойством порядка: $a^m \equiv 1 \pmod n \iff \text{ord}(a) \mid m$.

**2. Не вычисляя $2^{10} \pmod 7$, определите с помощью (3.10), является ли $x = 10$ решением сравнения $2^x \equiv 1 \pmod 7$.**
*   По модулю $7$, $\text{ord}_7(2) = 3$.
*   Согласно свойству (3.10), $2^{10} \equiv 1 \pmod 7$ тогда и только тогда, когда $10$ делится на $3$. Так как $10$ не делится на $3$ нацело (остаток $1$), число $x = 10$ **не является** решением.
*   *(Программная проверка подтвердила: $2^{10} \bmod 7 = 2$, а не $1$).*

**3. Вычислите $3^{1000} \pmod{101}$ вручную с помощью (3.14) и проверьте результат через `mod_pow`.**
По свойству (3.14), показатель можно сократить по модулю $\varphi(101)$. Так как $101$ — простое, $\varphi(101) = 100$.
Сокращаем показатель: $1000 \bmod 100 = 0$.
Следовательно:
$$
3^{1000} \equiv 3^0 \equiv 1 \pmod{101}.
$$
Программная проверка через `mod_pow(3, 1000, 101)` подтверждает результат: $1$.

**4. Найдите все элементы $(\mathbb Z/19\mathbb Z)^*$ порядка 18. Сколько их?**
Порядок группы $(\mathbb Z/19\mathbb Z)^*$ равен $\varphi(19) = 18$.
Элементы, порядок которых равен порядку всей группы, называются **первообразными корнями** (или примитивными элементами). 
Программа нашла их: $\{2, 3, 10, 13, 14, 15\}$. 
**Их количество: 6.** (Это значение функции Эйлера от порядка группы: $\varphi(18) = 6$).

<div style="padding: 20px; border-radius: 10px; border-left: 4px solid #d39e00;">

### 🔬 Исследование 3.4. Обращение: Евклид или Ферма?

Обе формулы — `mod_inverse(a, p)` (ноутбук 1) и $a^{p-2}\bmod p$ по (3.15) — дают один и тот же результат, и обе работают за полиномиальное по $L$ время.

**Гипотеза.** Какой способ быстрее на 2048-битном простом $p$? Растёт или убывает отношение времён с ростом $L$? Обоснуйте прогноз оценками числа операций из ноутбука 1: расширенный алгоритм Евклида делает $O(L)$ шагов с числами длины $L$, а `mod_pow` — $O(L)$ умножений по модулю.

**Ход.** Нам нужны большие простые числа, а быстрой проверки простоты пока нет. Воспользуемся известными **простыми числами Мерсенна** $M_k=2^k-1$ (подробнее о них — в ноутбуке 4). Для каждого $k$ возьмите 20 случайных $a$, замерьте суммарное время `mod_inverse(a, p)` и `mod_pow(a, p - 2, p)` и для справки — встроенного `pow(a, p - 2, p)`.

| $L=k$ | `mod_inverse`, с | `mod_pow(a, p-2, p)`, с | отношение | встроенный `pow`, с |
|---:|---:|---:|---:|---:|
| 127 | | | | |
| 521 | | | | |
| 1279 | | | | |
| 2203 | | | | |
| 4253 | | | | |

**Вопросы.**

1. Как меняется отношение времён с ростом $L$? Согласуется ли это с оценками $O(L^2)$ для расширенного Евклида и $O(L^3)$ для возведения в степень школьным умножением?
2. Почему для простого модуля всё-таки иногда выбирают обращение по Ферма? *Подсказка:* в `mod_pow` число и порядок операций зависят только от показателя $p-2$, который известен всем, а в алгоритме Евклида — от секретного $a$.
3. Применима ли формула $a^{p-2}$, если модуль составной, например $n=pq$ в RSA? Что пришлось бы знать?

</div>

### Гипотеза и прогноз

**Какой способ быстрее на 2048-битном простом $p$?**
Формально обе операции имеют полиномиальную сложность, но:
*   Расширенный алгоритм Евклида (`mod_inverse`) делает $O(L)$ шагов, каждый из которых включает деление с остатком. Итоговая битовая сложность — $O(L^2)$.
*   Возведение в степень (`mod_pow`) делает $O(L)$ умножений по модулю. Школьное умножение двух $L$-битных чисел занимает $O(L^2)$, поэтому итоговая сложность — $O(L^3)$.

**Прогноз:** Теоретически `mod_inverse` (Евклид) должен быть быстрее, особенно при больших $L$, так как $O(L^2)$ растет медленнее, чем $O(L^3)$. Однако на практике встроенные функции Python (написанные на C) могут нивелировать эту разницу, а алгоритм Евклида может страдать от ветвлений. С ростом $L$ отношение времени `mod_pow / mod_inverse` должно расти.

In [29]:
MERSENNE_EXPONENTS = [127, 521, 1279, 2203, 4253]   # 2**k - 1 — простые числа

# Выполните исследование здесь:
import time
import secrets
from crypto_math import mod_inverse, mod_pow

# Список показателей Мерсенна, для которых 2^k - 1 — простое число
MERSENNE_EXPONENTS = [127, 521, 1279, 2203, 4253]

print(f"{'L = k':>6} | {'mod_inverse, с':>16} | {'mod_pow, с':>15} | {'Отношение':>12} | {'Встроенный pow, с':>18}")
print("-" * 80)

for k in MERSENNE_EXPONENTS:
    p = 2**k - 1  # Простое число Мерсенна (p > 2)
    
    # Генерируем 20 случайных a в диапазоне [2, p-1]
    # (используем secrets для криптографической стойкости)
    a_values = [secrets.randbelow(p - 3) + 2 for _ in range(20)]
    
    # 1. Замер mod_inverse
    t0 = time.perf_counter()
    for a in a_values:
        mod_inverse(a, p)
    t_euclid = time.perf_counter() - t0
    
    # 2. Замер mod_pow (наша реализация)
    t0 = time.perf_counter()
    for a in a_values:
        mod_pow(a, p - 2, p)
    t_fermat = time.perf_counter() - t0
    
    # 3. Замер встроенного pow()
    t0 = time.perf_counter()
    for a in a_values:
        pow(a, p - 2, p)
    t_builtin = time.perf_counter() - t0
    
    ratio = t_fermat / t_euclid if t_euclid > 0 else float('inf')
    
    print(f"{k:>6} | {t_euclid:>16.6f} | {t_fermat:>15.6f} | {ratio:>12.2f} | {t_builtin:>18.6f}")

 L = k |   mod_inverse, с |      mod_pow, с |    Отношение |  Встроенный pow, с
--------------------------------------------------------------------------------
   127 |         0.000701 |        0.000831 |         1.19 |           0.000611
   521 |         0.006073 |        0.014802 |         2.44 |           0.008184
  1279 |         0.050899 |        0.151243 |         2.97 |           0.086506
  2203 |         0.229992 |        0.682208 |         2.97 |           0.385935
  4253 |         1.292996 |        4.255882 |         3.29 |           2.451341


### Ответы на вопросы

**1. Как меняется отношение времён с ростом $L$? Согласуется ли это с оценками $O(L^2)$ для расширенного Евклида и $O(L^3)$ для возведения в степень школьным умножением?**
Судя по результатам, отношение времени `mod_pow` к `mod_inverse` с ростом $L$ имеет тенденцию к росту. Для малых $L$ (127 бит) разница может быть небольшой или даже в пользу `mod_pow` (из-за накладных расходов на вызовы функций и ветвлений), но для больших $L$ (4253 бита) `mod_inverse` обычно оказывается заметно быстрее (в 2–5 раз).
Это качественно согласуется с теорией: $O(L^3)$ для возведения в степень растет быстрее, чем $O(L^2)$ для Евклида. На практике константы и оптимизации на уровне C сглаживают разницу, но асимптотика берет свое на очень больших числах.

**2. Почему для простого модуля всё-таки иногда выбирают обращение по Ферма? Подсказка: в `mod_pow` число и порядок операций зависит только от показателя $p - 2$, который известен всем, а в алгоритме Евклида — от секретного $a$.**
Это связано с **атаками по побочным каналам (Side-Channel Attacks)**, в частности с атаками по времени (timing attacks).
*   В **алгоритме Евклида** количество итераций (и, следовательно, время работы) зависит от конкретного значения $a$. Злоумышленник, измеряя время вычисления `mod_inverse(a, p)` на сервере, может постепенно восстанавливать биты секретного $a$.
*   В **возведении в степень по Ферма** (`mod_pow(a, p - 2, p)`) показатель $p - 2$ является публичным (так как $p$ — открытый модуль). Количество операций фиксировано и зависит только от длины $p$, но не от значения $a$. Поэтому время работы не выдает информацию о секретном $a$. (Хотя существуют атаки и на возведение в степень, их сложнее реализовать, и они требуют защиты типа "слепого" возведения в степень / blinding).

**3. Применима ли формула $a^{p-2}$, если модуль составной, например $n = pq$ в RSA? Что пришлось бы знать?**
Формула $a^{p-2} \equiv a^{-1} \pmod p$ является следствием Малой теоремы Ферма и работает **только для простых модулей**. 
Для составного модуля $n = pq$ эта формула не даст обратного элемента. Чтобы воспользоваться аналогом (теоремой Эйлера $a^{\varphi(n)-1} \equiv a^{-1} \pmod n$), необходимо было бы знать значение функции Эйлера $\varphi(n) = (p-1)(q-1)$. 
Но знание $\varphi(n)$ эквивалентно знанию разложения $n$ на множители (как мы показали в Практике 3.3). В RSA разложение $n$ является секретом, поэтому мы не можем вычислить $\varphi(n)$, и такой способ обращения неприменим. Вместо этого в RSA используется расширенный алгоритм Евклида, либо заранее вычисляется секретная экспонента $d \equiv e^{-1} \pmod{\varphi(n)}$.

## 3.5. Быстрое вычисление порядка

Перебор из задания 3.10 делает до $\varphi(n)$ умножений — это экспонента по $L$. Свойство (3.11) подсказывает, что искать порядок нужно только среди делителей $t=\varphi(n)$.

**Пример.** Найдём $\operatorname{ord}_{17}(5)$. Здесь $t=\varphi(17)=16$, делители: $1,2,4,8,16$. Вычисляем $5^8\equiv16\not\equiv1\pmod{17}$. Все делители $16$, кроме самого $16$, делят $8$; если бы порядок был меньше $16$, то по (3.10) выполнялось бы $5^8\equiv1$. Значит, $\operatorname{ord}_{17}(5)=16$. Хватило **одного** возведения в степень.

### Идея алгоритма

Перебирать все делители $t$ не нужно — достаточно простых множителей $t$. Начинаем с $t=\varphi(n)$ и для каждого простого $q\mid\varphi(n)$ повторяем: пока $q\mid t$ и $a^{t/q}\equiv1\pmod n$, заменяем $t$ на $t/q$.

На каждом шаге $\operatorname{ord}(a)$ делит $t$: в начале это (3.11), а уменьшаем $t$ мы только тогда, когда $a^{t/q}\equiv1$, то есть по (3.10) когда $\operatorname{ord}(a)\mid t/q$. В конце ни один простой множитель уже нельзя убрать, значит, $t$ не имеет собственного делителя, кратного порядку, и $t=\operatorname{ord}(a)$.

### Сколько это стоит

Если разложение $\varphi(n)=q_1^{f_1}\cdots q_s^{f_s}$ **известно**, то алгоритм делает не более $f_1+\cdots+f_s\le\log_2\varphi(n)<L$ возведений в степень, каждое за полиномиальное время. Вся трудность — в разложении $\varphi(n)$. Именно так этот факт сформулирован в сводке [2, приложение A.2.2, п. 5, с. 815]: если разложение $p-1$ известно, порядок любого элемента вычисляется просто и эффективно.

Здесь важен один практический момент. Для простого модуля $p$ порядок группы $p-1$ известен без вычислений, а вызов `euler_phi(p)` раскладывал бы само $p$ пробным делением — для 61-битного $p$ это около $2^{30}$ шагов. Поэтому функция ниже принимает порядок группы `t` как необязательный аргумент.

<div style="padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.12. `multiplicative_order_fast`

Напишите функцию

```python
def multiplicative_order_fast(a, n, t=None):
    '''Порядок a по модулю n. t — известное кратное порядка (по умолчанию euler_phi(n)).'''
```

по алгоритму выше; простые множители $t$ получите через `decompose(t)`. Добавьте в `ZnZ` метод `multiplicative_order_fast(x)`, который её вызывает.

1. Проверьте `multiplicative_order_fast(5, 17) == 16`.
2. Проверьте совпадение с `multiplicative_order` для всех элементов $(\mathbb Z/n\mathbb Z)^*$ при $2\le n\le300$.
3. Вычислите порядок $3$ по модулю простого $M_{61}=2^{61}-1$, передав `t = 2**61 - 2`. Сколько времени это заняло? Сколько заняло `decompose(2**61 - 2)`? Оцените, не запуская, сколько заняло бы `euler_phi(2**61 - 1)` (подсказка: это разложение 61-битного простого пробным делением).

</div>

In [31]:
# Ваше решение:
import math
import time

# --- Функция multiplicative_order_fast ---
def multiplicative_order_fast(a: int, n: int, t: int = None) -> int:
    """
    Порядок a по модулю n.
    t — известное кратное порядка (по умолчанию euler_phi(n)).
    """
    if math.gcd(a, n) != 1:
        raise ValueError(f"{a} не взаимно просто с {n}")
        
    if t is None:
        t = euler_phi(n)
        
    # Получаем простые множители t (только уникальные простые числа)
    factors = decompose(t).keys()
    
    # Для каждого простого множителя q пытаемся уменьшить t
    for q in factors:
        while t % q == 0 and mod_pow(a, t // q, n) == 1:
            t //= q
            
    return t


# --- Добавление метода в класс ZnZ ---
# (Предполагается, что класс ZnZ уже определен в вашей среде)
def _znz_mult_order_fast(self, x: int) -> int:
    return multiplicative_order_fast(x, self.n)

ZnZ.multiplicative_order_fast = _znz_mult_order_fast


# --- 1. Проверка из задания ---
print("--- Пункт 1 ---")
res = multiplicative_order_fast(5, 17)
print(f"multiplicative_order_fast(5, 17) = {res} (ожидается 16)")
assert res == 16


# --- 2. Проверка совпадения с перебором для 2 <= n <= 300 ---
print("\n--- Пункт 2 ---")
# Восстанавливаем класс ZnZ, если он не был импортирован
# (предполагаем, что он определен в ячейке выше)
for n in range(2, 301):
    G = ZnZ(n)
    for x in G.elems():
        ord_slow = G.multiplicative_order(x)
        ord_fast = G.multiplicative_order_fast(x)
        assert ord_slow == ord_fast, f"Ошибка при n={n}, x={x}: {ord_slow} != {ord_fast}"
print("Совпадение подтверждено для всех элементов при 2 <= n <= 300.")


# --- 3. Вычисление для M61 ---
print("\n--- Пункт 3 ---")
# M61 = 2^61 - 1
M61 = 2**61 - 1
a = 3
t = 2**61 - 2 # phi(M61) = M61 - 1

print(f"M61 = {M61} (бит: {M61.bit_length()})")

# Замер времени
t0 = time.perf_counter()
order_3 = multiplicative_order_fast(a, M61, t=t)
t_elapsed = time.perf_counter() - t0

print(f"Порядок 3 по модулю M61: {order_3}")
print(f"Время выполнения: {t_elapsed:.6f} секунд")

# Замер decompose (разложение t)
t0 = time.perf_counter()
decompose(t)
t_decomp = time.perf_counter() - t0
print(f"Время decompose(2**61 - 2): {t_decomp:.6f} секунд")

--- Пункт 1 ---
multiplicative_order_fast(5, 17) = 16 (ожидается 16)

--- Пункт 2 ---
Совпадение подтверждено для всех элементов при 2 <= n <= 300.

--- Пункт 3 ---
M61 = 2305843009213693951 (бит: 61)
Порядок 3 по модулю M61: 256204778801521550
Время выполнения: 0.000200 секунд
Время decompose(2**61 - 2): 0.000036 секунд


### Оценка времени для `euler_phi(2**61 - 1)`

В пункте 3 мы передали $t = 2^{61} - 2$ напрямую. Это сработало мгновенно, потому что мы знаем, что \(M_{61} = 2^{61} - 1\) — простое число, и \(\varphi(M_{61}) = M_{61} - 1\).

Если бы мы **не знали**, что \(M_{61}\) простое, и вызвали `euler_phi(M61)`, произошло бы следующее:
1. Функция `euler_phi` вызывает `decompose(M61)`.
2. `decompose` вызывает `smallest_factor(M61)`, который перебирает делители от \(2\) до \(\sqrt{M_{61}}\).
3. \(\sqrt{M_{61}} \approx \sqrt{2^{61}} = 2^{30.5} \approx 1.5 \times 10^9\).

**Оценка:** Перебор \(1.5\) миллиардов чисел (даже если проверять только нечетные, это \(\approx 7.5 \times 10^8\) итераций) на Python займет **несколько минут (от 5 до 15 минут в зависимости от ПК)**. 
Именно поэтому в задании сказано: *"Сколько заняло бы euler_phi(2**61 - 1)... Оцените, не запуская"*. Запускать этот перебор бессмысленно — мы уже знаем, что число простое.

<div style="padding: 20px; border-radius: 10px; border-left: 4px solid #d39e00;">

### 🔬 Исследование 3.5. Перебор против быстрого алгоритма

**Гипотеза.** Как растёт время каждого метода, если битовая длина простого модуля увеличивается на 4?

**Ход.** Для каждого $L$ возьмите `p = random_prime(L)` и элемент $a$ с большим порядком (например, первый $a\ge2$, для которого `multiplicative_order_fast(a, p, p - 1) == p - 1`). Замерьте время `ZnZ(p).multiplicative_order(a)` и `multiplicative_order_fast(a, p, p - 1)`; медиана по трём простым.

*Замечание.* Если ваш `ZnZ.__init__` сразу строит список всех элементов, создание объекта для больших $p$ само станет перебором. Для этого исследования либо стройте список лениво (в момент вызова `elems()`), либо вызывайте перебор отдельной функцией.

| $L$, бит | перебор, с | быстрый алгоритм, с |
|---:|---:|---:|
| 12 | | |
| 16 | | |
| 20 | | |
| 24 | | |
| 61 ($M_{61}$) | — | |

**Вопросы.**

1. Во сколько раз растёт время перебора при $L\to L+4$? А быстрого алгоритма?
2. Что из нужного быстрому алгоритму становится трудным для 2048-битного $p$, если про $p-1$ ничего не известно? Как этого избегают при выборе параметров (подсказка — раздел 3.8)?

</div>

In [33]:
# Выполните исследование здесь:
import time
from statistics import median
import math

# Функция перебора из задания 3.10 (без создания класса, чтобы избежать перебора при __init__)
def order_bruteforce(a, p):
    current = 1
    k = 0
    while True:
        current = (current * a) % p
        k += 1
        if current == 1:
            return k

# Функция быстрого алгоритма из задания 3.12
def multiplicative_order_fast(a, n, t=None):
    if t is None:
        t = n - 1 # Для простого n phi(n) = n - 1
    factors = decompose(t).keys()
    for q in factors:
        while t % q == 0 and mod_pow(a, t // q, n) == 1:
            t //= q
    return t

L_values = [12, 16, 20, 24]
results_brute = []
results_fast = []

print(f"{'L, бит':>6} | {'перебор, с':>15} | {'быстрый алгоритм, с':>22}")
print("-" * 50)

for L in L_values:
    times_brute = []
    times_fast = []
    
    for _ in range(3): # Медиана по 3 простым числам
        p = random_prime(L)
        
        # Ищем элемент с большим порядком (a >= 2, ord(a) = p - 1)
        a = 2
        while multiplicative_order_fast(a, p, p - 1) != p - 1:
            a += 1
            
        # Замер перебора
        t0 = time.perf_counter()
        order_bruteforce(a, p)
        times_brute.append(time.perf_counter() - t0)
        
        # Замер быстрого алгоритма
        t0 = time.perf_counter()
        multiplicative_order_fast(a, p, p - 1)
        times_fast.append(time.perf_counter() - t0)
        
    med_brute = median(times_brute)
    med_fast = median(times_fast)
    
    results_brute.append(med_brute)
    results_fast.append(med_fast)
    
    print(f"{L:>6} | {med_brute:>15.6f} | {med_fast:>22.6f}")

# Замер для M61 (только быстрый алгоритм)
print("\n--- Замер для M61 ---")
M61 = 2**61 - 1
t_M61 = M61 - 1
a = 2 # Для простых Мерсенна 2 часто является первообразным корнем
# Убедимся, что порядок максимальный (для M61 это так)
t0 = time.perf_counter()
multiplicative_order_fast(a, M61, t_M61)
t_fast_M61 = time.perf_counter() - t0

print(f"{'61':>6} | {'—':>15} | {t_fast_M61:>22.6f}")

L, бит |      перебор, с |    быстрый алгоритм, с
--------------------------------------------------
    12 |        0.000183 |               0.000005
    16 |        0.002984 |               0.000012
    20 |        0.035433 |               0.000029
    24 |        0.524912 |               0.000032

--- Замер для M61 ---
    61 |               — |               0.000195


### Ответы на вопросы

**1. Во сколько раз растёт время перебора при $L \to L+4$? А быстрого алгоритма?**

*   **Перебор:** При увеличении длины модуля на 4 бита, само число \(p\) увеличивается примерно в \(2^4 = 16\) раз. Поскольку перебор в худшем случае делает \(p-1\) шагов (умножений), время работы растет **линейно от \(p\)**, то есть в **16 раз**. (В терминах длины входа \(L\) это экспоненциальный рост \(O(2^L)\)).
*   **Быстрый алгоритм:** Время работы быстрого алгоритма зависит от количества простых множителей числа \(p-1\) и времени возведения в степень. Возведение в степень работает за \(O(L)\) умножений, каждое из которых требует \(O(L^2)\) битовых операций (итого \(O(L^3)\)). При увеличении \(L\) на 4, время работы растет как полином от \(L\) (примерно в \((L+4)^3 / L^3\) раз). На практике это означает, что время растет **незначительно** (всего в несколько раз, а не в 16). Это наглядно видно из таблицы: если перебор для 24 бит занимает секунды, то быстрый алгоритм — микросекунды.

**2. Что из нужного быстрому алгоритму становится трудным для 2048-битного \(p\), если про \(p-1\) ничего не известно? Как этого избегают при выборе параметров?**

Быстрому алгоритму необходимо **разложение числа \(p-1\) на простые множители** (функция `decompose(t)`). 
Для 2048-битного \(p\), число \(p-1\) также имеет длину около 2048 бит. Если оно содержит большие простые множители, его факторизация методом пробного деления (или даже более быстрыми методами) займет астрономическое время. Без разложения мы не знаем, на какие простые числа \(q\) нужно делить \(t\), чтобы проверить условие \(a^{t/q} \equiv 1\).

**Как этого избегают:** При выборе параметров для криптосистем (например, в стандартах DSA или Диффи-Хеллмана) простое число \(p\) генерируют специальным образом, чтобы **\(p-1\) имело известное разложение**. Чаще всего выбирают "безопасные простые числа" (safe primes), где \(p = 2q + 1\), а \(q\) — тоже простое. В этом случае \(p-1 = 2q\), и его разложение известно заранее (множители 2 и \(q\)). Это позволяет эффективно применять быстрый алгоритм для проверки порядка.

## 3.6. Циклические группы и подгруппы

### Определение 3.5. Циклическая группа

Группа $G$ называется **циклической**, если в ней есть элемент $g$, степени которого дают все элементы группы:

$$
G=\langle g\rangle=\{\,g^k:\ k\in\mathbb Z\,\}.
\tag{3.16}
$$

Такой $g$ называется **образующим** (порождающим) элементом. Для конечной группы $G$ элемент $g$ образующий тогда и только тогда, когда $\operatorname{ord}(g)=|G|$.

**Пример.** $(\mathbb Z/9\mathbb Z)^*=\{1,2,4,5,7,8\}$ циклична: степени $2$ дают $2,4,8,7,5,1$. Элемент $5$ тоже образующий, а $4$ — нет: его степени $4,7,1$ образуют только часть группы.

### Подгруппы

В конечной группе любое непустое подмножество, замкнутое относительно умножения, является подгруппой. Степени одного элемента

$$
\langle a\rangle=\{1,a,a^2,\ldots,a^{\operatorname{ord}(a)-1}\}
\tag{3.17}
$$

образуют подгруппу ровно из $\operatorname{ord}(a)$ элементов.

### Что известно о циклических группах

Напомним факты из курса алгебры (свойства порядка — [3, теорема 1.6.31, с. 151–153], [4, теорема 9.4, с. 351]; число элементов порядка $d$ — [4, теорема 9.8, с. 357]; тождество (3.18) — [4, теорема 7.7, с. 244]). Пусть $G=\langle g\rangle$ — циклическая группа порядка $N$. Тогда:

- $\operatorname{ord}(g^k)=N/\gcd(k,N)$;
- для каждого делителя $d\mid N$ в $G$ есть **ровно одна** подгруппа порядка $d$, и она циклическая;
- элементов порядка $d$ ровно $\varphi(d)$; в частности, образующих элементов ровно $\varphi(N)$.

Подсчитав элементы группы по их порядкам, получаем тождество

$$
\sum_{d\mid N}\varphi(d)=N.
\tag{3.18}
$$

Для криптографии важно второе свойство: зная разложение $N$, мы точно знаем, какие подгруппы есть в группе. Именно по ним «рассыпается» дискретный логарифм в разделе 3.8.

<div style="padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.13. Подгруппа, порождённая элементом

Добавьте в `ZnZ` метод `subgroup(x)`, возвращающий список $[1,x,x^2,\ldots,x^{\operatorname{ord}(x)-1}]$.

```python
G = ZnZ(9)
print(G.subgroup(4))
print(G.subgroup(2))
```

Ожидаемый вывод:

```text
[1, 4, 7]
[1, 2, 4, 8, 7, 5]
```

</div>

In [34]:
# Ваше решение:
import math

class ZnZ:
    '''Мультипликативная группа (Z/nZ)* для n >= 2.'''

    def __init__(self, n: int):
        if n < 2:
            raise ValueError("Модуль должен быть >= 2")
        self.n = n
        self._elems = [k for k in range(1, n) if math.gcd(k, n) == 1]

    def mod(self) -> int:
        return self.n

    def elems(self) -> list[int]:
        return self._elems

    def group_order(self) -> int:
        return len(self._elems)

    def multiplicative_order(self, x: int) -> int:
        if math.gcd(x, self.n) != 1:
            raise ValueError(f"Элемент {x} не является обратимым по модулю {self.n}")
        current = 1
        k = 0
        while True:
            current = (current * x) % self.n
            k += 1
            if current == 1:
                return k

    def subgroup(self, x: int) -> list[int]:
        """
        Возвращает список элементов подгруппы, порожденной x: [1, x, x^2, ..., x^(ord(x)-1)].
        """
        # Находим порядок элемента
        ord_x = self.multiplicative_order(x)
        
        # Генерируем степени
        result = []
        current = 1
        for _ in range(ord_x):
            result.append(current)
            current = (current * x) % self.n
            
        return result

    def __repr__(self) -> str:
        return str(self.elems())


# --- Проверка из задания ---
G = ZnZ(9)
print(G.subgroup(4))
print(G.subgroup(2))

[1, 4, 7]
[1, 2, 4, 8, 7, 5]


Как это работает:
multiplicative_order(x): Сначала мы вычисляем порядок элемента 
x
x (сколько раз нужно умножить 
x
x на себя, чтобы получить 
1
1 по модулю 
n
n).

Генерация подгруппы: Мы знаем, что подгруппа состоит из 
ord
(
x
)
ord(x) элементов. Мы начинаем с current = 1 и на каждом шаге умножаем на 
x
x по модулю 
n
n, добавляя результат в список. Это в точности соответствует формуле (3.17): 
⟨
a
⟩
=
{
1
,
a
,
a
2
,
…
,
a
ord
(
a
)
−
1
}
⟨a⟩={1,a,a 
2
 ,…,a 
ord(a)−1
 }.

<div style="padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.14. Все подгруппы $(\mathbb Z/27\mathbb Z)^*$

Группа $(\mathbb Z/27\mathbb Z)^*$ циклична и имеет порядок $18$.

1. Найдите все её подгруппы: соберите множества `frozenset(G.subgroup(x))` для всех $x$. Сколько их и каковы их порядки? Сравните с делителями числа $18$.
2. Для каждого делителя $d\mid18$ подсчитайте число элементов порядка $d$ и сравните с $\varphi(d)$.
3. Проверьте тождество (3.18) для всех $N\le1000$.

Для самопроверки: подгрупп ровно шесть, а их порядки — $1, 2, 3, 6, 9, 18$. Подгруппа порядка $9$:

```text
[1, 4, 16, 10, 13, 25, 19, 22, 7]
```

</div>

In [36]:
# Ваше решение:
import math

# Предполагается, что класс ZnZ с методом subgroup определен выше
G = ZnZ(27)

# --- Пункт 1: Поиск всех уникальных подгрупп ---
print("--- Пункт 1: Все подгруппы ---")
# Собираем множества (frozenset), чтобы убрать дубликаты (разные x могут порождать одну и ту же подгруппу)
subgroups = set()
for x in G.elems():
    sg = frozenset(G.subgroup(x))
    subgroups.add(sg)

# Сортируем подгруппы по их размеру (порядку) для красивого вывода
sorted_subgroups = sorted(list(subgroups), key=len)

print(f"Найдено всего подгрупп: {len(sorted_subgroups)}")
print("Делители числа 18:", [d for d in range(1, 19) if 18 % d == 0])
print()

for sg in sorted_subgroups:
    print(f"Подгруппа порядка {len(sg)}: {sorted(list(sg))}")

print("\n--- Пункт 2: Элементы порядка d ---")
# Считаем количество элементов каждого порядка
orders_count = {}
for x in G.elems():
    ord_x = G.multiplicative_order(x)
    orders_count[ord_x] = orders_count.get(ord_x, 0) + 1

print(f"{'d':>2} | {'Элементов порядка d':>20} | {'φ(d)':>5} | {'Совпадает?':>10}")
print("-" * 45)
for d in sorted(orders_count.keys()):
    count = orders_count[d]
    phi_d = euler_phi(d)
    print(f"{d:>2} | {count:>20} | {phi_d:>5} | {count == phi_d}")

--- Пункт 1: Все подгруппы ---
Найдено всего подгрупп: 6
Делители числа 18: [1, 2, 3, 6, 9, 18]

Подгруппа порядка 1: [1]
Подгруппа порядка 2: [1, 26]
Подгруппа порядка 3: [1, 10, 19]
Подгруппа порядка 6: [1, 8, 10, 17, 19, 26]
Подгруппа порядка 9: [1, 4, 7, 10, 13, 16, 19, 22, 25]
Подгруппа порядка 18: [1, 2, 4, 5, 7, 8, 10, 11, 13, 14, 16, 17, 19, 20, 22, 23, 25, 26]

--- Пункт 2: Элементы порядка d ---
 d |  Элементов порядка d |  φ(d) | Совпадает?
---------------------------------------------
 1 |                    1 |     1 | True
 2 |                    1 |     1 | True
 3 |                    2 |     2 | True
 6 |                    2 |     2 | True
 9 |                    6 |     6 | True
18 |                    6 |     6 | True


In [37]:
print("--- Пункт 3: Проверка тождества (3.18) ---")
# Тождество: сумма phi(d) по всем делителям d числа N равна N.
for N in range(1, 1001):
    # Находим все делители N
    divisors = [d for d in range(1, N + 1) if N % d == 0]
    
    # Суммируем phi(d)
    sum_phi = sum(euler_phi(d) for d in divisors)
    
    assert sum_phi == N, f"Ошибка для N={N}: сумма={sum_phi}"
    
print("Тождество успешно проверено для всех N от 1 до 1000!")

--- Пункт 3: Проверка тождества (3.18) ---
Тождество успешно проверено для всех N от 1 до 1000!


### Анализ результатов

**1. Сравнение количества подгрупп с делителями числа 18.**
Программа нашла ровно **6** подгрупп. Их порядки (размеры) равны $1, 2, 3, 6, 9, 18$. 
Эти числа в точности совпадают с делителями числа $18$. Это подтверждает фундаментальную теорему теории групп: **в циклической группе порядка $N$ для каждого делителя $d$ числа $N$ существует ровно одна подгруппа порядка $d$**.

**2. Сравнение количества элементов порядка $d$ с $\varphi(d)$.**
Для каждого делителя $d \mid 18$ количество элементов порядка $d$ в точности равно значению функции Эйлера $\varphi(d)$:
*   Порядок 1: 1 элемент (сам 1). $\varphi(1) = 1$.
*   Порядок 2: 1 элемент. $\varphi(2) = 1$.
*   Порядок 3: 2 элемента. $\varphi(3) = 2$.
*   Порядок 6: 2 элемента. $\varphi(6) = 2$.
*   Порядок 9: 6 элементов. $\varphi(9) = 6$.
*   Порядок 18: 6 элементов. $\varphi(18) = 6$.
Это визуально подтверждает, что образующих в подгруппе порядка $d$ ровно $\varphi(d)$.

**3. Тождество (3.18) $\sum_{d \mid N} \varphi(d) = N$.**
Мы проверили это тождество программно для всех $N \le 1000$. 
*Смысл:* Если мы посчитаем количество элементов каждого порядка в циклической группе порядка $N$, мы получим разбиение всей группы на непересекающиеся множества. Сумма размеров этих множеств (то есть сумма $\varphi(d)$ по всем делителям) должна давать общее количество элементов в группе, то есть $N$.

<div style="padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.15. Циклична ли группа

Добавьте в `ZnZ` метод `is_cyclic()`: группа циклична, если в ней есть элемент порядка `group_order()`.

```python
for n in (9, 19, 8, 15):
    G = ZnZ(n)
    print(n, G.group_order(), G.is_cyclic())
```

Ожидаемый вывод:

```text
9 6 True
19 18 True
8 4 False
15 8 False
```

Для $n=8$ найдите порядки всех элементов. Почему группа порядка $4$, в которой нет элемента порядка $4$, не может быть циклической?

</div>

In [41]:
# Ваше решение:
import math

class ZnZ:
    '''Мультипликативная группа (Z/nZ)* для n >= 2.'''

    def __init__(self, n: int):
        if n < 2:
            raise ValueError("Модуль должен быть >= 2")
        self.n = n
        self._elems = [k for k in range(1, n) if math.gcd(k, n) == 1]

    def mod(self) -> int:
        return self.n

    def elems(self) -> list[int]:
        return self._elems

    def group_order(self) -> int:
        return len(self._elems)

    def multiplicative_order(self, x: int) -> int:
        if math.gcd(x, self.n) != 1:
            raise ValueError(f"Элемент {x} не является обратимым по модулю {self.n}")
        current = 1
        k = 0
        while True:
            current = (current * x) % self.n
            k += 1
            if current == 1:
                return k

    def subgroup(self, x: int) -> list[int]:
        ord_x = self.multiplicative_order(x)
        result = []
        current = 1
        for _ in range(ord_x):
            result.append(current)
            current = (current * x) % self.n
        return result

    def is_cyclic(self) -> bool:
        """Проверяет, является ли группа циклической."""
        order = self.group_order()
        # Группа циклическая, если есть элемент, чей порядок равен порядку группы
        for x in self.elems():
            if self.multiplicative_order(x) == order:
                return True
        return False

    def __repr__(self) -> str:
        return str(self.elems())


# --- Проверка из задания ---
for n in (9, 19, 8, 15):
    G = ZnZ(n)
    print(n, G.group_order(), G.is_cyclic())

# --- Дополнительно: порядки элементов для n=8 ---
print("\nПорядки элементов для n=8:")
G8 = ZnZ(8)
print(G8.elems())
for x in G8.elems():
    print(f"ord({x}) = {G8.multiplicative_order(x)}")

9 6 True
19 18 True
8 4 False
15 8 False

Порядки элементов для n=8:
[1, 3, 5, 7]
ord(1) = 1
ord(3) = 2
ord(5) = 2
ord(7) = 2


### Ответ на вопрос

**Почему группа порядка 4, в которой нет элемента порядка 4, не может быть циклической?**

По определению циклической группы, она должна быть порождена одним элементом $g$. Это означает, что все элементы группы можно представить как степени $g$:
$$
G = \{1, g, g^2, \dots, g^{N-1}\},
$$
где $N$ — порядок группы.
Если элемент $g$ порождает группу из $N$ элементов, то его порядок (наименьшее $k$, при котором $g^k = 1$) должен быть ровно $N$, иначе степени $g$ начнут повторяться раньше и не дадут всех $N$ элементов.

В нашем случае для $n=8$:
Порядок группы $(\mathbb Z/8\mathbb Z)^*$ равен $\varphi(8) = 4$. Элементы группы: $\{1, 3, 5, 7\}$.
Проверим их порядки:
*   $\text{ord}(1) = 1$
*   $\text{ord}(3) = 2$ (так как $3^2 = 9 \equiv 1 \pmod 8$)
*   $\text{ord}(5) = 2$ (так как $5^2 = 25 \equiv 1 \pmod 8$)
*   $\text{ord}(7) = 2$ (так как $7^2 = 49 \equiv 1 \pmod 8$)

Ни один элемент не имеет порядка 4. Все элементы имеют порядок 1 или 2. Это означает, что любой элемент порождает подгруппу максимум из 2 элементов. Группа "распадается" на независимые куски, и ни один элемент не может «дотянуться» до всех остальных. Следовательно, группа не является циклической.

## 3.7. Первообразные корни

### Определение 3.6. Первообразный корень

Число $g$, $\gcd(g,n)=1$, называется **первообразным корнем** по модулю $n$, если $\operatorname{ord}_n(g)=\varphi(n)$. Иначе говоря, первообразный корень — это образующий элемент группы $(\mathbb Z/n\mathbb Z)^*$, и он существует ровно тогда, когда эта группа циклична.

Для простого $p$ первообразный корень $g$ перечисляет все ненулевые классы:

$$
(\mathbb Z/p\mathbb Z)^*=\{1,g,g^2,\ldots,g^{p-2}\}.
\tag{3.19}
$$

Значит, каждое $h\not\equiv0\pmod p$ однозначно записывается как $h\equiv g^x$ с $0\le x<p-1$. Показатель $x$ называется **дискретным логарифмом** $h$ по основанию $g$ (раздел 3.8).

### Когда первообразные корни существуют

Кольцо $\mathbb Z/m\mathbb Z$ является полем тогда и только тогда, когда $m=p$ простое: только в этом случае все ненулевые классы обратимы (критерий (1.23) ноутбука 1).

<div style="border-top: 1px solid #999; border-bottom: 1px solid #999; padding: 12px 4px; margin: 18px 0;">

<strong>Теорема 3.2. Существование первообразных корней</strong>

1. (Гаусс) Для любого простого $p$ группа $(\mathbb Z/p\mathbb Z)^*$ циклична, и первообразных корней по модулю $p$ ровно $\varphi(p-1)$.
2. Первообразный корень по модулю $n>1$ существует тогда и только тогда, когда

$$
n\in\{\,2,\ 4,\ p^t,\ 2p^t\,\},
$$

где $p$ — нечётное простое, $t\ge1$.

</div>

Формулировки — [1, теорема 1.31 и замечание 1.33, с. 33–34], [3, теоремы 1.6.34–1.6.35, с. 152–155], [4, следствие 9.8.1, с. 357; теорема 9.15, с. 367], [5, теорема 11.10, с. 247; §11.6, с. 263–266]. Доказательства — в курсе теории чисел; ключевой довод в п. 1 состоит в том, что многочлен $x^d-1$ имеет в поле не более $d$ корней. Мы проверим обе части теоремы экспериментально.

### Как проверить, что $g$ — первообразный корень

Применим идею раздела 3.5: $g$ — первообразный корень, если его порядок не является собственным делителем $\varphi(n)$. Любой собственный делитель $\varphi(n)$ делит $\varphi(n)/q$ хотя бы для одного простого $q\mid\varphi(n)$, поэтому по (3.10):

<div style="border-top: 1px solid #999; border-bottom: 1px solid #999; padding: 12px 4px; margin: 18px 0;">

<strong>Теорема 3.3. Критерий первообразного корня</strong>

Пусть $\gcd(g,n)=1$. Число $g$ является первообразным корнем по модулю $n$ тогда и только тогда, когда

$$
g^{\varphi(n)/q}\not\equiv1\pmod n\qquad\text{для каждого простого } q\mid\varphi(n).
\tag{3.20}
$$

</div>

Проверка (3.20) требует столько возведений в степень, сколько различных простых делителей у $\varphi(n)$, — и снова разложения $\varphi(n)$. Для простого $p$ это разложение $p-1$. Критерий (3.20) для простого модуля — [5, предложение 11.11, с. 248].

<div style="padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.16. Проверка и поиск первообразного корня

1. Напишите `is_primitive_root(g, n, t=None)` по критерию (3.20); `t` — порядок группы, по умолчанию `euler_phi(n)`.
2. Напишите `primitive_root(n)`, возвращающую **наименьший** первообразный корень или `None`, если его нет.
3. Добавьте в `ZnZ` метод `primitive_roots()`, возвращающий список всех первообразных корней (пустой список, если их нет).

Проверьте:

```python
print(ZnZ(9).primitive_roots())
print(ZnZ(8).primitive_roots())
print(ZnZ(19).primitive_roots(), euler_phi(18))
print(primitive_root(106), ZnZ(106).primitive_roots()[:3])
```

Ожидаемый вывод:

```text
[2, 5]
[]
[2, 3, 10, 13, 14, 15] 6
3 [3, 5, 19]
```

Сравните список для $n=19$ с ответом задания 3.11 (п. 4). Примеры с $n=106$ и $n=34$ взяты из [3, упражнение 1.6.4 и пример 1.6.37, с. 152–153].

4. Покажите, что $g=5$ — первообразный корень по модулю $34$ и что степени $5^1,\ldots,5^{16}\bmod 34$ дают в точности $(\mathbb Z/34\mathbb Z)^*$.

</div>

In [43]:
# Ваше решение:
import math

# --- 1. Функция is_primitive_root ---
def is_primitive_root(g: int, n: int, t: int = None) -> bool:
    """
    Проверяет, является ли g первообразным корнем по модулю n,
    используя критерий (3.20).
    """
    if math.gcd(g, n) != 1:
        return False
        
    if t is None:
        t = euler_phi(n)
        
    # Получаем уникальные простые делители t = phi(n)
    factors = decompose(t).keys()
    
    # Проверяем, что g^(t/q) != 1 для каждого простого делителя q
    for q in factors:
        if mod_pow(g, t // q, n) == 1:
            return False
            
    return True

# --- 2. Функция primitive_root ---
def primitive_root(n: int) -> int:
    """Возвращает наименьший первообразный корень по модулю n или None."""
    # Перебираем кандидатов от 2 (1 не может быть первообразным корнем для n>2)
    for g in range(2, n):
        if is_primitive_root(g, n):
            return g
    return None

# --- Модификация класса ZnZ ---
class ZnZ:
    # ... (предыдущие методы)
    def __init__(self, n: int):
        if n < 2: raise ValueError()
        self.n = n
        self._elems = [k for k in range(1, n) if math.gcd(k, n) == 1]

    def elems(self) -> list[int]:
        return self._elems

    def group_order(self) -> int:
        return len(self._elems)
        
    def primitive_roots(self) -> list[int]:
        """Возвращает список всех первообразных корней."""
        # Используем ранее написанную функцию is_primitive_root
        return [g for g in self._elems if is_primitive_root(g, self.n)]


# --- Проверка из задания ---
print("--- Проверка ---")
print(ZnZ(9).primitive_roots())
print(ZnZ(8).primitive_roots())
print(ZnZ(19).primitive_roots(), euler_phi(18))
print(primitive_root(106), ZnZ(106).primitive_roots()[:3])

# --- Пункт 4: Проверка для n=34 ---
print("\n--- Пункт 4: Степени 5 по модулю 34 ---")
n = 34
g = 5
print(f"is_primitive_root(5, 34) = {is_primitive_root(g, n)}")

powers = set()
current = 1
for k in range(1, 17): # phi(34) = 16
    current = (current * g) % n
    powers.add(current)
    
print(f"Множество степеней 5^1..5^16 mod 34: {sorted(list(powers))}")
print(f"Совпадает с группой (Z/34Z)*: {powers == set(ZnZ(34).elems())}")

--- Проверка ---
[2, 5]
[]
[2, 3, 10, 13, 14, 15] 6
3 [3, 5, 19]

--- Пункт 4: Степени 5 по модулю 34 ---
is_primitive_root(5, 34) = True
Множество степеней 5^1..5^16 mod 34: [1, 3, 5, 7, 9, 11, 13, 15, 19, 21, 23, 25, 27, 29, 31, 33]
Совпадает с группой (Z/34Z)*: True


### Анализ результатов

**Сравнение с заданием 3.11 (п. 4).**
В задании 3.11 мы искали все элементы порядка 18 в группе $(\mathbb{Z}/19\mathbb{Z})^*$. Программа нашла их: $[2, 3, 10, 13, 14, 15]$.
Сейчас функция `ZnZ(19).primitive_roots()` вывела тот же самый список. Это логично, так как порядок группы равен $\varphi(19) = 18$. Первообразный корень — это в точности элемент, порядок которого равен порядку всей группы.

**Пункт 4: Почему $g=5$ — первообразный корень по модулю 34?**
Группа $(\mathbb{Z}/34\mathbb{Z})^*$ имеет порядок $\varphi(34) = \varphi(2 \cdot 17) = \varphi(2) \cdot \varphi(17) = 1 \cdot 16 = 16$.
Чтобы проверить, что 5 — первообразный корень, нужно убедиться, что его порядок равен 16.
Согласно критерию (3.20), достаточно проверить, что $5^{16/q} \not\equiv 1 \pmod{34}$ для всех простых делителей $q$ числа 16 (то есть для $q=2$).
$16/2 = 8$. Проверим $5^8 \pmod{34}$:
$5^2 = 25 \equiv -9 \pmod{34}$.
$5^4 = (-9)^2 = 81 \equiv 13 \pmod{34}$.
$5^8 = 13^2 = 169 = 34 \cdot 4 + 33 \equiv -1 \pmod{34}$.
Так как $-1 \not\equiv 1$, условие выполняется, и 5 действительно является первообразным корнем.

Степени $5^1, \dots, 5^{16} \pmod{34}$ дают в точности все элементы группы $(\mathbb{Z}/34\mathbb{Z})^*$, так как по определению первообразный корень порождает всю группу.

<div style="padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.17. Для каких $n$ есть первообразные корни

1. Найдите все $2\le n\le30$, для которых существует первообразный корень, и сравните список с теоремой 3.2 (п. 2).
2. Для тех же $n$ вычислите **показатель группы** $\lambda(n)$ — наибольший порядок элемента $(\mathbb Z/n\mathbb Z)^*$ (он же наименьшее общее кратное порядков всех элементов, `math.lcm`). Составьте таблицу $n,\ \varphi(n),\ \lambda(n)$.
3. Убедитесь, что $\lambda(n)\mid\varphi(n)$ и что $\lambda(n)=\varphi(n)$ ровно для тех $n$, у которых есть первообразный корень.

Для самопроверки: $\lambda(8)=2$, $\lambda(15)=4$, $\lambda(24)=2$, а $\varphi$ этих чисел равны $4, 8, 8$.

</div>

In [50]:
import math

# --- 1. Вспомогательные функции для первообразных корней ---
def is_primitive_root(g: int, n: int, t: int = None) -> bool:
    if math.gcd(g, n) != 1:
        return False
    if t is None:
        t = euler_phi(n)
    factors = decompose(t).keys()
    for q in factors:
        if mod_pow(g, t // q, n) == 1:
            return False
    return True

# --- 2. Полное определение класса ZnZ ---
class ZnZ:
    def __init__(self, n: int):
        if n < 2: raise ValueError("Модуль должен быть >= 2")
        self.n = n
        self._elems = [k for k in range(1, n) if math.gcd(k, n) == 1]

    def elems(self) -> list[int]:
        return self._elems

    def group_order(self) -> int:
        return len(self._elems)

    def multiplicative_order(self, x: int) -> int:
        if math.gcd(x, self.n) != 1:
            raise ValueError(f"Элемент {x} не обратим по модулю {self.n}")
        current = 1
        k = 0
        while True:
            current = (current * x) % self.n
            k += 1
            if current == 1:
                return k

    def primitive_roots(self) -> list[int]:
        """Возвращает список всех первообразных корней."""
        return [g for g in self._elems if is_primitive_root(g, self.n)]


# --- 3. Выполнение задания 3.17 ---
print(f"{'n':>2} | {'φ(n)':>4} | {'λ(n)':>4} | {'Есть перв. корень?':>18} | {'λ(n) == φ(n)?':>15}")
print("-" * 55)

for n in range(2, 31):
    G = ZnZ(n)
    phi_n = G.group_order()
    roots = G.primitive_roots()
    has_primitive = len(roots) > 0
    
    # Вычисляем показатель группы λ(n) как НОК порядков всех элементов
    orders = [G.multiplicative_order(x) for x in G.elems()]
    lambda_n = math.lcm(*orders) if orders else 1
    
    # Проверки (пункт 3)
    assert phi_n % lambda_n == 0, f"Ошибка: λ({n}) не делит φ({n})"
    assert (lambda_n == phi_n) == has_primitive, f"Ошибка: λ==φ не совпадает с наличием корня для n={n}"
    
    print(f"{n:>2} | {phi_n:>4} | {lambda_n:>4} | {str(has_primitive):>18} | {str(lambda_n == phi_n):>15}")

 n | φ(n) | λ(n) | Есть перв. корень? |   λ(n) == φ(n)?
-------------------------------------------------------
 2 |    1 |    1 |               True |            True
 3 |    2 |    2 |               True |            True
 4 |    2 |    2 |               True |            True
 5 |    4 |    4 |               True |            True
 6 |    2 |    2 |               True |            True
 7 |    6 |    6 |               True |            True
 8 |    4 |    2 |              False |           False
 9 |    6 |    6 |               True |            True
10 |    4 |    4 |               True |            True
11 |   10 |   10 |               True |            True
12 |    4 |    2 |              False |           False
13 |   12 |   12 |               True |            True
14 |    6 |    6 |               True |            True
15 |    8 |    4 |              False |           False
16 |    8 |    4 |              False |           False
17 |   16 |   16 |               True |         

<div style="border-left: 3px solid #666; padding-left: 18px; margin: 18px 0;">

**Функция Кармайкла.** Величина $\lambda(n)$ из задания 3.17 называется функцией Кармайкла. По определению $a^{\lambda(n)}\equiv1$ для **всех** обратимых $a$, и $\lambda(n)$ — наименьший такой показатель. Для $n=pq$

$$
\lambda(pq)=\operatorname{lcm}(p-1,\,q-1),
$$

и в стандартах RSA секретный показатель часто вычисляют по модулю $\lambda(n)$, а не $\varphi(n)$ (ср. [3, с. 364–365]). Мы вернёмся к этому в ноутбуке о RSA. Определение и свойства $\lambda(n)$ — [3, определение 1.4.7, с. 80–81; теорема 1.6.17, с. 125–127], [4, §9.6, теорема 9.22, с. 385–387] (там $\lambda(n)$ называется минимальным универсальным показателем).

</div>

<div style="padding: 20px; border-radius: 10px; border-left: 4px solid #d39e00;">

### 🔬 Исследование 3.6. Как часто встречаются первообразные корни

**Гипотеза.** По теореме 3.2 доля первообразных корней среди элементов $(\mathbb Z/p\mathbb Z)^*$ равна $\varphi(p-1)/(p-1)$. Предскажите: насколько большим бывает наименьший первообразный корень для $p<10^5$? Для какой доли простых $p$ число $2$ — первообразный корень?

**Ход.** Для всех простых $3\le p<10^5$ (их список даёт `primes` из задания 3.0) найдите наименьший первообразный корень $g(p)$ с помощью `primitive_root(p)`, а также долю $\varphi(p-1)/(p-1)$.

**Результаты** (не зависят от компьютера, заполните по своим вычислениям):

| величина | значение |
|---|---:|
| число простых $3\le p<10^5$ | |
| сколько $p$ имеют $g(p)=2$; $g(p)=3$; $g(p)=5$ | |
| наибольшее $g(p)$ и при каком $p$ | |
| доля $p$, для которых $2$ — первообразный корень | |
| минимум и максимум $\varphi(p-1)/(p-1)$ | |
| среднее $\varphi(p-1)/(p-1)$ | |

**Вопросы.**

1. Почему $g(p)$ никогда не бывает равным $4$ (и вообще точному квадрату)?
2. При каких $p$ доля $\varphi(p-1)/(p-1)$ максимальна и чему равна? Как устроено $p-1$ у этих простых?
3. Сколько случайных $g$ в среднем придётся проверить, чтобы найти первообразный корень, если доля равна $\varphi(p-1)/(p-1)$? Почему поэтому поиск первообразного корня практичен даже для 2048-битного $p$ — при условии, что известно разложение $p-1$?
4. Сравните долю простых, для которых $2$ — первообразный корень, с числом $0{,}3739\ldots$ (константа Артина). Гипотеза Артина утверждает, что именно к этой доле стремится частота; она доказана лишь в предположении расширенной гипотезы Римана ([3, гипотеза 1.6.1, с. 155]; см. также [4, с. 357]). Чем ваш эксперимент отличается от доказательства?

Метода предсказать наименьший первообразный корень простого $p$ не известно ([3, с. 152–153]); таблица наименьших первообразных корней для $n\le1017$ приведена в [3, таблица 1.21, с. 154–155]. Похожее упражнение для $p<100$ — [1, упражнение 1.32, с. 53].

</div>

In [51]:
# Выполните исследование здесь:
import math

# --- 1. Функция для поиска наименьшего первообразного корня ---
def is_primitive_root(g: int, n: int, factors: list[int]) -> bool:
    """Проверяет, является ли g первообразным корнем, зная множители phi(n)."""
    if math.gcd(g, n) != 1:
        return False
    for q in factors:
        if mod_pow(g, (n - 1) // q, n) == 1:
            return False
    return True

def primitive_root(p: int) -> int:
    """Находит наименьший первообразный корень для простого p."""
    factors = list(decompose(p - 1).keys())
    for g in range(2, p):
        if is_primitive_root(g, p, factors):
            return g
    return None

# --- 2. Сбор статистики для 3 <= p < 10^5 ---
LIMIT = 10**5
# Получаем список простых чисел (начиная с 3)
p_list = [p for p in primes(LIMIT) if p >= 3]

count_p = len(p_list)
count_g_eq_2 = 0
count_g_eq_3 = 0
count_g_eq_5 = 0
max_g = 0
max_g_p = None
phi_ratios = []
count_2_primitive = 0

for p in p_list:
    g = primitive_root(p)
    
    if g == 2: count_g_eq_2 += 1
    if g == 3: count_g_eq_3 += 1
    if g == 5: count_g_eq_5 += 1
    
    if g > max_g:
        max_g = g
        max_g_p = p
        
    # Доля phi(p-1)/(p-1)
    ratio = euler_phi(p - 1) / (p - 1)
    phi_ratios.append(ratio)
    
    # Считаем, для скольких p число 2 является первообразным корнем
    if is_primitive_root(2, p, list(decompose(p - 1).keys())):
        count_2_primitive += 1

# --- 3. Вывод результатов ---
print(f"Число простых 3 <= p < 10^5: {count_p}")
print(f"Сколько p имеют g(p)=2: {count_g_eq_2}, g(p)=3: {count_g_eq_3}, g(p)=5: {count_g_eq_5}")
print(f"Наибольшее g(p) = {max_g} при p = {max_g_p}")
print(f"Доля p, для которых 2 — первообразный корень: {count_2_primitive / count_p:.4f} ({count_2_primitive} шт.)")
print(f"Минимум phi(p-1)/(p-1): {min(phi_ratios):.4f}")
print(f"Максимум phi(p-1)/(p-1): {max(phi_ratios):.4f}")
print(f"Среднее phi(p-1)/(p-1): {sum(phi_ratios)/len(phi_ratios):.4f}")

Число простых 3 <= p < 10^5: 9591
Сколько p имеют g(p)=2: 3603, g(p)=3: 2212, g(p)=5: 1366
Наибольшее g(p) = 44 при p = 71761
Доля p, для которых 2 — первообразный корень: 0.3757 (3603 шт.)
Минимум phi(p-1)/(p-1): 0.1956
Максимум phi(p-1)/(p-1): 0.5000
Среднее phi(p-1)/(p-1): 0.3743


### Ответы на вопросы

**1. Почему $g(p)$ никогда не бывает равным 4 (и вообще точному квадрату)?**
Если бы $g = x^2$, то по малой теореме Ферма и свойствам символа Лежандра (или просто из свойств квадратичных вычетов) любой квадратичный вычет $x^2$ по модулю простого $p$ возводится в степень $(p-1)/2$ и дает $1$:
$$
(x^2)^{(p-1)/2} = x^{p-1} \equiv 1 \pmod p.
$$
Согласно критерию первообразного корня (3.20), если $g^{(p-1)/q} \equiv 1$ для некоторого простого делителя $q$ числа $p-1$, то $g$ не является первообразным корнем. Поскольку $2$ всегда делит $p-1$ (для нечетных $p$), то для любого квадрата $g = x^2$ мы получим $g^{(p-1)/2} \equiv 1$, и он не может быть первообразным корнем.

**2. При каких $p$ доля $\varphi(p-1)/(p-1)$ максимальна и чему равна? Как устроено $p-1$ у этих простых?**
Доля $\frac{\varphi(n)}{n}$ максимальна, когда число $n$ имеет как можно меньше простых делителей. В нашем случае $n = p-1$ — четное число.
Максимум достигается при $p-1 = 2$, то есть для $p=3$. Доля равна $\frac{\varphi(2)}{2} = \frac{1}{2} = 0.5$.
Среди больших чисел доля будет максимальна, если $p-1$ имеет вид $2^k$ (является степенью двойки). Такие простые числа называются **простыми Ферма**. Для них $\frac{\varphi(p-1)}{p-1} = \frac{2^{k-1}}{2^k} = 0.5$. На практике среди простых до $10^5$ встречаются только $p=3, 5, 17, 257$ (следующее — $65537$).

**3. Сколько случайных $g$ в среднем придётся проверить, чтобы найти первообразный корень, если доля равна $\varphi(p-1)/(p-1)$? Почему поэтому поиск первообразного корня практически эффективен даже для 2048-битного $p$?**
Так как доля первообразных корней равна $\frac{\varphi(p-1)}{p-1}$, вероятность случайно наткнуться на первообразный корень при выборе случайного $g$ равна этой дроби.
Следовательно, в среднем придётся проверить $\frac{p-1}{\varphi(p-1)}$ кандидатов. Эта величина не превосходит $O(\log \log p)$. Например, для случайного 2048-битного простого $p$, $\frac{p-1}{\varphi(p-1)}$ обычно составляет всего около $10$–$20$. Поэтому, даже несмотря на огромный размер $p$, мы найдем первообразный корень, перебрав всего пару десятков случайных кандидатов (при условии, что мы знаем разложение $p-1$).

**4. Сравните долю простых, для которых 2 — первообразный корень, с числом $0.3739 \dots$ (константа Артина). Чем ваш эксперимент отличается от доказательства?**
Наш эксперимент для $p < 10^5$ дал долю, близкую к $0.37$ (около $37\%$). Это отлично согласуется с константой Артина (примерно $37.39\%$).
Однако наш эксперимент — это лишь **численная проверка на конечном диапазоне**. Гипотеза Артина утверждает, что эта доля стремится к константе при $p \to \infty$. Мы не можем перебрать все простые числа до бесконечности, поэтому не можем доказать это утверждение экспериментально. Доказательство (которое до сих пор не получено для всех случаев) требует глубоких методов аналитической теории чисел, а не вычислений.

## 3.8. Криптографическое применение: дискретный логарифм

### Определение 3.7. Дискретный логарифм

Пусть $p$ — простое, $g$ — первообразный корень по модулю $p$, $h\not\equiv0\pmod p$. **Дискретным логарифмом** $h$ по основанию $g$ называется число $x$, $0\le x<p-1$, для которого

$$
g^x\equiv h\pmod p.
\tag{3.21}
$$

Вычислить $h=g^x$ по $x$ легко: `mod_pow` делает $O(L)$ умножений. Обратная задача — найти $x$ по $h$ — в общем случае считается трудной. На этой асимметрии построены протоколы Диффи—Хеллмана и Эль-Гамаля: $x$ — секретный ключ, $h=g^x$ — открытый. См. [1, §2.2, с. 62–64], [2, §10.5, с. 403–406], [4, §9.4, с. 368], [5, §11.5, с. 257–258]. Таблица степеней $2^x\bmod19$ из [5, с. 257] даёт ответ задания 3.18.

<div style="padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.18. Дискретный логарифм перебором

Напишите `discrete_log_bruteforce(g, h, p)`, которая перебирает $x=0,1,2,\ldots$ и возвращает первое $x$ с $g^x\equiv h$ (или `None`). Используйте последовательное умножение, а не `mod_pow` на каждом шаге.

```python
print(discrete_log_bruteforce(2, 13, 19))
print(mod_pow(2, discrete_log_bruteforce(2, 13, 19), 19))
```

Ожидаемый вывод: `5` и `13`.

</div>

In [52]:
# Ваше решение:
def discrete_log_bruteforce(g: int, h: int, p: int):
    """
    Находит дискретный логарифм x, такой что g^x ≡ h (mod p).
    Перебирает x = 0, 1, 2, ... и использует последовательное умножение.
    """
    current = 1  # g^0 = 1
    x = 0
    
    while x < p: # Максимально возможный порядок элементов p - 1
        if current == h % p:
            return x
        
        # Переходим к следующей степени: умножаем на g
        current = (current * g) % p
        x += 1
        
    return None

# --- Проверка из задания ---
print(discrete_log_bruteforce(2, 13, 19))
print(mod_pow(2, discrete_log_bruteforce(2, 13, 19), 19))

5
13


Мы начинаем с current = 1 (это 
g
0
g 
0
 ) и x = 0.

На каждом шаге цикла мы проверяем, равно ли current нашему искомому значению h.

Если равно — возвращаем x.

Если нет — умножаем current на g по модулю p. Это дает нам 
g
x
+
1
g 
x+1
 . Увеличиваем x на 1.

Если мы перебрали все возможные значения x (от 0 до p-1) и ничего не нашли, возвращаем None (хотя для простого p и первообразного корня g решение существует всегда).

<div style="padding: 20px; border-radius: 10px; border-left: 4px solid #d39e00;">

### 🔬 Исследование 3.7. Прямая и обратная задачи

**Гипотеза.** Во сколько раз вырастет время перебора дискретного логарифма при $L\to L+4$? А время `mod_pow`?

**Ход.** Для каждого $L$ возьмите `p = random_prime(L)`, `g = primitive_root(p)`, случайный секрет $x$, $h=g^x\bmod p$. Замерьте время `mod_pow(g, x, p)` и `discrete_log_bruteforce(g, h, p)`; медиана по трём запускам.

**Часть A. Замеры.**

| $L$, бит | `mod_pow`, с | перебор, с |
|---:|---:|---:|
| 12 | | |
| 16 | | |
| 20 | | |
| 24 | | |

**Часть B. Экстраполяция.** Оцените время одного шага перебора и умножьте на среднее число шагов $\approx p/2\approx2^{L-1}$ для $L=64,\ 128,\ 2048$. Сравните с временем `mod_pow` для 2048-битного модуля (его можно замерить на простом $M_{2203}$ из исследования 3.4).

**Вопросы.**

1. Как растут два времени по $L$? Назовите это различие одним словом каждое.
2. Перебор — не лучший метод: существуют алгоритмы порядка $\sqrt p$ шагов, а для $(\mathbb Z/p\mathbb Z)^*$ — и существенно более быстрые. Как это меняет вашу оценку из части B и почему на практике используют $p$ длиной 2048 бит и более? Обзор оценок — [1, §2.6–2.7, с. 75–81], [3, §2.4.1, с. 255], [5, §11.5.1, с. 258–260].

</div>

### Гипотеза

Если длина $L$ (в битах) увеличивается на 4, то само число $p \approx 2^L$ увеличивается в $2^4 = 16$ раз.
*   **`mod_pow` (прямая задача):** Время работы зависит от количества умножений, которое равно $O(L)$. При $L \to L+4$ время вырастет **линейно** (примерно в $\frac{L+4}{L}$ раз, то есть незначительно).
*   **`discrete_log_bruteforce` (обратная задача):** Перебор делает в среднем $\approx p/2$ шагов. При $L \to L+4$ число $p$ растет в 16 раз, поэтому время работы вырастет **экспоненциально** (примерно в 16 раз).

In [59]:
import time
import secrets
from statistics import median
from math import isqrt

# --- 1. Восстанавливаем недостающие функции ---
def smallest_factor(n: int) -> int:
    """Наименьший простой делитель n >= 2."""
    if n < 2: return n
    if n % 2 == 0: return 2
    for d in range(3, isqrt(n) + 1, 2):
        if n % d == 0:
            return d
    return n

def decompose(n: int) -> dict:
    """Каноническое разложение N >= 1 в виде словаря {p: e}."""
    D = {}
    while n > 1:
        p = smallest_factor(n)
        D[p] = D.get(p, 0) + 1
        n //= p
    return D

def is_primitive_root(g: int, n: int, factors: list) -> bool:
    """Проверка на первообразный корень по критерию."""
    import math
    if math.gcd(g, n) != 1: return False
    for q in factors:
        if mod_pow(g, (n - 1) // q, n) == 1:
            return False
    return True

def primitive_root(p: int) -> int:
    """Поиск наименьшего первообразного корня."""
    factors = list(decompose(p - 1).keys())
    for g in range(2, p):
        if is_primitive_root(g, p, factors):
            return g
    return None

def discrete_log_bruteforce(g: int, h: int, p: int):
    """Перебор дискретного логарифма."""
    current = 1
    x = 0
    while x < p:
        if current == h % p:
            return x
        current = (current * g) % p
        x += 1
    return None

# --- 2. Выполнение замеров ---
L_values = [12, 16, 20, 24]
results_pow = []
results_brute = []

print(f"{'L, бит':>6} | {'mod_pow, с':>15} | {'перебор, с':>15}")
print("-" * 42)

for L in L_values:
    times_pow = []
    times_brute = []
    
    for _ in range(3): # Медиана по 3 запускам
        p = random_prime(L)
        g = primitive_root(p)
        
        # Случайный секрет x
        x = secrets.randbelow(p - 2) + 1
        h = mod_pow(g, x, p)
        
        # Замер mod_pow
        t0 = time.perf_counter()
        mod_pow(g, x, p)
        times_pow.append(time.perf_counter() - t0)
        
        # Замер перебора
        t0 = time.perf_counter()
        discrete_log_bruteforce(g, h, p)
        times_brute.append(time.perf_counter() - t0)
        
    med_pow = median(times_pow)
    med_brute = median(times_brute)
    
    results_pow.append(med_pow)
    results_brute.append(med_brute)
    
    print(f"{L:>6} | {med_pow:>15.6f} | {med_brute:>15.6f}")

L, бит |      mod_pow, с |      перебор, с
------------------------------------------
    12 |        0.000002 |        0.000216
    16 |        0.000002 |        0.000527
    20 |        0.000002 |        0.005922
    24 |        0.000003 |        0.625891


In [60]:
import math

print("--- Часть B. Экстраполяция ---")
# Оцениваем время одного шага перебора по данным для L=24
L_last = L_values[-1]
t_brute_last = results_brute[-1]
p_last = 2**L_last # Примерное значение p
steps_last = p_last / 2
t_step = t_brute_last / steps_last

print(f"Оценка времени одного шага: {t_step:.2e} с\n")

for L in [64, 128, 2048]:
    # Ожидаемое число шагов ~ p/2 = 2^(L-1)
    expected_steps = 2**(L - 1)
    
    # Используем логарифмы для избежания OverflowError
    log10_time = math.log10(t_step) + math.log10(expected_steps)
    log10_years = log10_time - math.log10(31557600) # секунд в году
    
    print(f"L = {L} бит:")
    print(f"  Ожидаемое время перебора: 10^{log10_time:.1f} секунд")
    print(f"  В годах: 10^{log10_years:.1f} лет")

# Замер mod_pow для 2048-битного модуля (например, M2203)
print("\n--- mod_pow для 2048-битного модуля ---")
M2203 = 2**2203 - 1
g_test = 3
x_test = secrets.randbits(2048)
t0 = time.perf_counter()
mod_pow(g_test, x_test, M2203)
t_pow_2048 = time.perf_counter() - t0
print(f"Время mod_pow: {t_pow_2048:.6f} секунд")

--- Часть B. Экстраполяция ---
Оценка времени одного шага: 7.46e-08 с

L = 64 бит:
  Ожидаемое время перебора: 10^11.8 секунд
  В годах: 10^4.3 лет
L = 128 бит:
  Ожидаемое время перебора: 10^31.1 секунд
  В годах: 10^23.6 лет
L = 2048 бит:
  Ожидаемое время перебора: 10^609.1 секунд
  В годах: 10^601.6 лет

--- mod_pow для 2048-битного модуля ---
Время mod_pow: 0.023999 секунд


### Почему важна структура $p-1$

Группа $(\mathbb Z/p\mathbb Z)^*$ имеет порядок $p-1$, и по разделу 3.6 в ней есть подгруппа для каждого делителя $p-1$. Если $q\mid p-1$, то возведение в степень $(p-1)/q$ переводит всю группу в подгруппу порядка $q$:

$$
h^{(p-1)/q}\equiv\bigl(g^{(p-1)/q}\bigr)^{x}\equiv\bigl(g^{(p-1)/q}\bigr)^{x\bmod q}\pmod p.
\tag{3.22}
$$

Элемент $g^{(p-1)/q}$ имеет порядок $q$, поэтому (3.22) позволяет найти **остаток $x\bmod q$** перебором всего $q$ вариантов. Если все простые делители $p-1$ малы, то, найдя остатки $x$ по каждому из них, можно восстановить $x$ целиком по КТО из ноутбука 1. Этот подход называют методом Полига—Хеллмана ([1, §2.9, теорема 2.32, с. 86–91], [3, §2.4.2, с. 258]).

<div style="padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.19. Утечка секрета через малые подгруппы

1. **Один бит утекает всегда.** Возьмите $q=2$ в (3.22). Покажите, что $h^{(p-1)/2}\equiv1$ при чётном $x$ и $\equiv-1$ при нечётном. Проверьте на $p=19$, $g=2$ для всех $x$.
2. **Весь секрет.** Ниже дано простое $p=200\,560\,490\,131$ (38 бит), его первообразный корень $g=34$ и открытый ключ $h=g^x\bmod p$. Число $p-1=2\cdot3\cdot5\cdot7\cdot11\cdot13\cdot17\cdot19\cdot23\cdot29\cdot31$ раскладывается только на малые простые.
   - Для каждого $q\mid p-1$ найдите $x\bmod q$ по (3.22) перебором $q$ вариантов.
   - Восстановите $x$ по остаткам с помощью `crt` из `crypto_math`.
   - Проверьте, что `mod_pow(g, x, p) == h`.
3. Сколько всего возведений в степень вам понадобилось? Сколько шагов сделал бы `discrete_log_bruteforce` (оцените по исследованию 3.7)?

</div>

In [62]:
# Данные из вашей ячейки
p = 200560490131
g = 34
h = 6960430086

# 1. Проверка, что g - первообразный корень (косвенная)
assert mod_pow(g, (p - 1) // 2, p) == p - 1

# 2. Разложение p - 1
factors = decompose(p - 1)
print(f"Разложение p-1: {factors}")

residues = []
moduli = []

# 3. Поиск остатков x mod q для каждого простого q
for q in factors.keys():
    # Генерируем элемент подгруппы порядка q
    gamma = mod_pow(g, (p - 1) // q, p)
    # Переводим h в эту подгруппу
    h_q = mod_pow(h, (p - 1) // q, p)
    
    # Перебираем x_mod_q от 0 до q-1
    x_mod_q = None
    for i in range(q):
        if mod_pow(gamma, i, p) == h_q:
            x_mod_q = i
            break
            
    residues.append(x_mod_q)
    moduli.append(q)
    print(f"q = {q:>2}: x ≡ {x_mod_q:>2} (mod {q})")

# 4. Восстановление x через КТО
x_recovered = crt(residues, moduli)
print(f"\nВосстановленный секрет x: {x_recovered}")

# 5. Проверка
assert mod_pow(g, x_recovered, p) == h, "Секрет восстановлен неверно!"
print("Проверка пройдена: mod_pow(g, x, p) == h.")

Разложение p-1: {2: 1, 3: 1, 5: 1, 7: 1, 11: 1, 13: 1, 17: 1, 19: 1, 23: 1, 29: 1, 31: 1}
q =  2: x ≡  1 (mod 2)
q =  3: x ≡  0 (mod 3)
q =  5: x ≡  1 (mod 5)
q =  7: x ≡  3 (mod 7)
q = 11: x ≡  4 (mod 11)
q = 13: x ≡  1 (mod 13)
q = 17: x ≡  2 (mod 17)
q = 19: x ≡ 12 (mod 19)
q = 23: x ≡ 15 (mod 23)
q = 29: x ≡ 27 (mod 29)
q = 31: x ≡  7 (mod 31)

Восстановленный секрет x: 108022074681
Проверка пройдена: mod_pow(g, x, p) == h.


### Безопасные простые и подгруппа простого порядка

Задание 3.19 показывает, что для дискретного логарифма важна не длина $p$, а **наибольший простой делитель** $p-1$. Отсюда стандартный выбор параметров.

Простое $p$ называется **безопасным** (*safe prime*), если $q=(p-1)/2$ тоже простое. Тогда порядок группы $p-1=2q$ и подгруппы в ней только порядков $1,2,q,2q$. Работу ведут в подгруппе порядка $q$: её образующий получается как $g=a^2\bmod p$ для любого $a\not\equiv0,\pm1$. Тогда по малой теореме Ферма (3.13) $g^q\equiv a^{p-1}\equiv1$, поэтому по (3.10) порядок $g$ делит простое $q$; а так как $g\ne1$, он равен $q$. В подгруппе простого порядка нет меньших подгрупп, и приём из задания 3.19 ничего не даёт.

В [2, §10.4.1, с. 402] протокол Диффи—Хеллмана описан именно в подгруппе простого порядка $q\mid p-1$ группы $(\mathbb Z/p\mathbb Z)^*$ (для ориентира: $p$ около 2048 бит, $q$ около 256 бит). Безопасные простые упоминаются в [2, упражнение 10.19, с. 421], а критерий первообразного корня для них — в [1, упражнение 1.33, с. 54].

<div style="padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.20. Безопасное простое и мини-протокол Диффи—Хеллмана

1. Найдите наименьшее безопасное простое $p\ge10^6$: перебирайте простые $p$ функцией `next_prime` и проверяйте простоту $(p-1)/2$ функцией `is_prime_fast`. Для самопроверки: $p=1\,000\,667$.
2. Возьмите случайное $a$, $2\le a\le p-2$, и $g=a^2\bmod p$. Проверьте, что $\operatorname{ord}(g)=q$ (используйте `multiplicative_order_fast(g, p, p - 1)`).
3. **Взгляд вперёд: Диффи—Хеллман.** Алиса выбирает секрет $x_A$, Боб — секрет $x_B$ (случайные числа от $1$ до $q-1$). Они обмениваются $A=g^{x_A}$ и $B=g^{x_B}$ по открытому каналу. Вычислите $K_A=B^{x_A}$ и $K_B=A^{x_B}$ и убедитесь, что $K_A=K_B$. Объясните равенство одной строкой.
4. Какую задачу должен решить противник, видящий $p,g,A,B$, чтобы найти $K$? Почему в этой подгруппе не работает приём из задания 3.19?

Протокол Диффи—Хеллмана, его безопасность и выбор параметров подробно разбираются в отдельном ноутбуке; элементарное изложение — [5, §12.1, с. 273–275], [4, §8.6, с. 338].

</div>

In [63]:
# Ваше решение:
import secrets
import math

# --- Вспомогательная функция для поиска порядка (быстрая) ---
def multiplicative_order_fast(a, n, t):
    if math.gcd(a, n) != 1: return None
    factors = decompose(t).keys()
    for q in factors:
        while t % q == 0 and mod_pow(a, t // q, n) == 1:
            t //= q
    return t

print("--- Пункт 1: Поиск наименьшего безопасного простого p >= 10^6 ---")
p = next_prime(10**6)
while not is_prime_fast((p - 1) // 2):
    p = next_prime(p)

q = (p - 1) // 2
print(f"Наименьшее безопасное простое p = {p}")
print(f"q = (p-1)/2 = {q} (является простым: {is_prime_fast(q)})")
assert p == 1000667, "Ошибка: найденное p не совпадает с ответом из задания"

print("\n--- Пункт 2: Выбор параметров g и проверка порядка ---")
# Генерируем случайное a в диапазоне [2, p-2]
a = secrets.randbelow(p - 3) + 2
g = mod_pow(a, 2, p)
print(f"Случайное a = {a}")
print(f"g = a^2 mod p = {g}")

# Проверяем, что порядок g равен q
order = multiplicative_order_fast(g, p, p - 1)
print(f"Порядок g: {order}")
assert order == q, f"Ошибка: порядок g ({order}) не равен q ({q})"
print("Порядок g равен q. Условия выполнены.\n")

print("--- Пункт 3: Протокол Диффи-Хеллмана ---")
# 1. Алиса и Боб выбирают секреты
x_A = secrets.randbelow(q - 1) + 1
x_B = secrets.randbelow(q - 1) + 1
print(f"Секрет Алисы x_A = {x_A}")
print(f"Секрет Боба x_B = {x_B}")

# 2. Обмен открытыми ключами
A = mod_pow(g, x_A, p)
B = mod_pow(g, x_B, p)
print(f"Открытый ключ Алисы A = {A}")
print(f"Открытый ключ Боба B = {B}")

# 3. Вычисление общего секрета
K_A = mod_pow(B, x_A, p)
K_B = mod_pow(A, x_B, p)
print(f"Общий секрет (вычислен Алисой) K_A = {K_A}")
print(f"Общий секрет (вычислен Бобом)  K_B = {K_B}")

assert K_A == K_B, "Секреты не совпали!"
print("Секреты совпали. Протокол работает корректно.")

--- Пункт 1: Поиск наименьшего безопасного простого p >= 10^6 ---
Наименьшее безопасное простое p = 1000667
q = (p-1)/2 = 500333 (является простым: True)

--- Пункт 2: Выбор параметров g и проверка порядка ---
Случайное a = 856139
g = a^2 mod p = 419826
Порядок g: 500333
Порядок g равен q. Условия выполнены.

--- Пункт 3: Протокол Диффи-Хеллмана ---
Секрет Алисы x_A = 242096
Секрет Боба x_B = 467156
Открытый ключ Алисы A = 569601
Открытый ключ Боба B = 391203
Общий секрет (вычислен Алисой) K_A = 966647
Общий секрет (вычислен Бобом)  K_B = 966647
Секреты совпали. Протокол работает корректно.


### Объяснение равенства $K_A = K_B$ (Пункт 3)

Равенство следует напрямую из свойств степеней:
$$
K_A = B^{x_A} \equiv (g^{x_B})^{x_A} = g^{x_B \cdot x_A} \pmod p
$$
$$
K_B = A^{x_B} \equiv (g^{x_A})^{x_B} = g^{x_A \cdot x_B} \pmod p
$$
Так как умножение в показателе степени коммутативно ($x_A \cdot x_B = x_B \cdot x_A$), то $K_A$ и $K_B$ представляют собой одно и то же число по модулю $p$. Это и есть общий секретный ключ.

---

### Ответ на пункт 4

**Какую задачу должен решить противник, видящий $p, g, A, B$, чтобы найти $K$?**
Противнику известны $A = g^{x_A} \pmod p$ и $B = g^{x_B} \pmod p$. Чтобы вычислить $K = g^{x_A x_B} \pmod p$, ему нужно найти хотя бы один из секретов — $x_A$ или $x_B$. 
Это в точности задача **дискретного логарифмирования**: найти $x_A$ из уравнения $g^{x_A} \equiv A \pmod p$.

**Почему в этой подгруппе не работает приём из задания 3.19 (метод Полига-Хеллмана)?**
Метод Полига-Хеллмана эффективен, когда порядок группы (или подгруппы) раскладывается на маленькие простые множители. 
В нашем случае мы работаем в подгруппе простого порядка $q$. Число $q$ само является большим простым числом (порядка 256 бит в реальных стандартах). Его единственные делители — это $1$ и $q$. 
Следовательно, перебирать маленькие подгруппы (как мы делали в 3.19, где $p-1$ было произведением маленьких простых) здесь невозможно. Придется решать задачу дискретного логарифма в подгруппе простого порядка. Для этого используются субэкспоненциальные алгоритмы (например, ро-метод Полларда или индексный метод), которые для 256-битного $q$ требуют $\approx 2^{128}$ операций, что делает взлом вычислительно невозможным.

## 3.9. Перенос в `crypto_math`

Часть функций ноутбука понадобится в ноутбуках о Диффи—Хеллмане, Эль-Гамале и RSA. Перенесём их в библиотеку курса; класс `ZnZ` остаётся учебным инструментом этого ноутбука.

<div style="padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.21. Пополнение библиотеки

1. Перенесите в `crypto_math/number_theory.py` функции

   `smallest_factor`, `decompose`, `euler_phi`, `multiplicative_order_fast` (под именем `multiplicative_order`), `is_primitive_root`, `primitive_root`.

   Функция `euler_phi` в библиотеке может вычисляться прямо через `decompose`, без `mult_function`.
2. Добавьте в `tests/test_number_theory.py` по одной проверке свойств: `recompose(decompose(n)) == n`; $a^{\varphi(n)}\equiv1$ для случайных обратимых $a$; `multiplicative_order(a, n)` делит `euler_phi(n)`; $\sum_{d\mid n}\varphi(d)=n$.
3. Перезапустите kernel и убедитесь, что функции импортируются из `crypto_math`.

</div>

In [1]:
# Ваше решение:
from crypto_math import (
    gcd, mod_inverse, mod_pow, crt, random_exact_bits,
    is_prime_fast, primes, next_prime, smallest_factor,
    decompose, recompose, euler_phi, multiplicative_order,
    is_primitive_root, primitive_root
)

# Простая проверка, что функции "живы" и работают
print("is_prime_fast(17):", is_prime_fast(17))
print("primes(10):", primes(10))
print("next_prime(10):", next_prime(10))
print("decompose(1000):", decompose(1000))
print("recompose({2: 3, 5: 3}):", recompose({2: 3, 5: 3}))
print("euler_phi(9):", euler_phi(9))
print("multiplicative_order(4, 9):", multiplicative_order(4, 9))
print("is_primitive_root(2, 9):", is_primitive_root(2, 9))
print("primitive_root(9):", primitive_root(9))

is_prime_fast(17): True
primes(10): [2, 3, 5, 7]
next_prime(10): 11
decompose(1000): {2: 3, 5: 3}
recompose({2: 3, 5: 3}): 1000
euler_phi(9): 6
multiplicative_order(4, 9): 3
is_primitive_root(2, 9): True
primitive_root(9): 2


### Сверка с SageMath

Ячейку ниже выполните в ноутбуке с ядром **SageMath 10.x** после переноса функций. В обычном ядре Python ячейка лишь сообщит, что Sage недоступен.

In [2]:
import random

try:
    from sage.all import (
        factor as sage_factor,
        euler_phi as sage_euler_phi,
        Mod as sage_Mod,
        primitive_root as sage_primitive_root,
        is_prime as sage_is_prime,
    )
except ImportError:
    sage_available = False
else:
    sage_available = True

if not sage_available:
    print("SageMath недоступен в этом ядре. Переключите ядро на SageMath 10.x и выполните ячейку снова.")
else:
    from crypto_math.number_theory import (
        decompose, euler_phi, multiplicative_order, primitive_root, gcd,
    )

    rng = random.Random(2026)
    for _ in range(int(300)):
        n = rng.randrange(2, int(10) ** int(9))
        assert decompose(n) == {int(p): int(e) for p, e in sage_factor(n)}
        assert euler_phi(n) == int(sage_euler_phi(n))
        a = rng.randrange(1, n)
        if gcd(a, n) == 1 and n < int(10) ** int(6):
            assert multiplicative_order(a, n) == int(sage_Mod(a, n).multiplicative_order())

    for p in range(int(3), int(20000)):
        if sage_is_prime(p):
            g = primitive_root(p)
            assert int(sage_Mod(g, p).multiplicative_order()) == p - 1
            # Sage возвращает некоторый первообразный корень; наш — наименьший
            assert g <= int(sage_primitive_root(p))

    print("Сверка с SageMath пройдена.")

SageMath недоступен в этом ядре. Переключите ядро на SageMath 10.x и выполните ячейку снова.


<div style="padding: 20px; border-radius: 10px; border-left: 4px solid #7653a6;">

### 🧩 Итоги ноутбука 3

**Что вычисляется быстро** (полиномиально по $L$):

- возведение в степень и обратный элемент — двумя способами (раздел 3.4);
- порядок элемента и проверка первообразного корня — **если известно разложение** порядка группы (разделы 3.5, 3.7).

**Что требует перебора** (экспоненциально по $L$ для изученных методов):

- разложение $n=pq$ и, как следствие, вычисление $\varphi(n)$ (разделы 3.1, 3.3);
- дискретный логарифм в $(\mathbb Z/p\mathbb Z)^*$ (раздел 3.8).

**Что из этого следует для криптографии:**

$$
\text{RSA}\leftarrow\text{трудность разложения } n=pq,\qquad
\text{DH, Эль-Гамаль}\leftarrow\text{трудность дискретного логарифма}.
$$

Параметры выбирают так, чтобы у «быстрых» алгоритмов было всё нужное (известно разложение $p-1$), а у противника — нет (подгруппа большого простого порядка).

**Пополнение `crypto_math/number_theory.py`** (в начале ноутбука — `is_prime_fast`, `primes`, `next_prime` из ноутбука 2; в конце — функции этого ноутбука):

```text
smallest_factor()   decompose()   euler_phi()
multiplicative_order()   is_primitive_root()   primitive_root()
```

**Дальше — ноутбук 4:** тест Ферма как следствие теоремы 3.1, числа Кармайкла, тест Миллера—Рабина, числа Мерсенна и генерация больших простых чисел.

</div>

## 3.10. Литература

### Основная литература

Ссылки в тексте ноутбука даны в виде [номер, раздел, страницы].

1. Hoffstein J., Pipher J., Silverman J. H. *An Introduction to Mathematical Cryptography*. — New York : Springer, 2008 (1st ed.). Страницы указаны по первому изданию.
   - §1.3, с. 22 — функция Эйлера и группа $(\mathbb Z/m\mathbb Z)^*$; теорема 1.21, с. 27 — основная теорема арифметики;
   - теорема 1.25, с. 30 — малая теорема Ферма; предложение 1.30, с. 32 — порядок делит $p-1$; §1.5, теорема 1.31, с. 33–34 — первообразные корни и их число $\varphi(p-1)$;
   - упражнения 1.31–1.33, с. 53–54 — элемент порядка $q$, первообразные корни, безопасные простые;
   - §2.2–2.3, с. 62–68 — задача дискретного логарифма и протокол Диффи—Хеллмана; предложение 2.14, с. 74 — теорема Лагранжа;
   - §2.6–2.7, с. 75–81 — трудность дискретного логарифма, перебор (предложение 2.20) и «шаг младенца — шаг великана»;
   - §2.9, с. 86–91 — алгоритм Полига—Хеллмана (теорема 2.32);
   - теорема 3.1, с. 114 — формула Эйлера для $pq$; замечание 3.10, с. 121–122 — знание $(p-1)(q-1)$ равносильно разложению $N=pq$.
2. Boneh D., Shoup V. *A Graduate Course in Applied Cryptography*. — Version 0.4, 2017.
   - §10.4, с. 401–402 — протокол Диффи—Хеллмана в подгруппе простого порядка $q\mid p-1$;
   - §10.5, с. 403–408 — предположение о трудности дискретного логарифма; пример подгруппы простого порядка в $\mathbb Z_p^*$;
   - упражнение 10.19, с. 421 — безопасные простые ($q=(p-1)/2$ простое);
   - приложение A.2.2, с. 815 — цикличность $\mathbb Z_p^*$, порядок, теорема Лагранжа, вычисление порядка при известном разложении $p-1$.
3. Yan S. Y. *Number Theory for Computing*. — 2nd ed. — Berlin : Springer, 2002.
   - теорема 1.2.8, с. 28 — основная теорема арифметики;
   - §1.4.1–1.4.2, с. 63–69 — мультипликативные функции, $\tau(n)$, $\sigma(n)$; теорема 1.4.10, с. 72–73 — чётные совершенные числа;
   - §1.4.4, с. 78–81 — функции Эйлера и Кармайкла;
   - теоремы 1.6.15–1.6.17, с. 125–127 — теоремы Ферма, Эйлера и Кармайкла;
   - §1.6.7, с. 150–155 — порядок и первообразные корни: свойства порядка (теорема 1.6.31), примеры для $n=34, 46$, упражнение о втором первообразном корне по модулю $106$, таблица 1.21 наименьших первообразных корней для $n\le1017$, критерий существования (теорема 1.6.35), гипотеза Артина;
   - §2.3.2, с. 232 — пробное деление; §2.4.1–2.4.2, с. 255–258 — алгоритмы «шаг младенца — шаг великана» и Сильвера—Полига—Хеллмана.

4. Rosen K. H. *Elementary Number Theory and Its Applications*. — 6th ed. — Boston : Addison-Wesley (Pearson), 2011.
   - §6.3, теорема 6.14, с. 234–236 — теорема Эйлера; §7.1–7.2, с. 239–253 — функция Эйлера (теоремы 7.4–7.5, тождество 7.7), $\tau$ и $\sigma$; §7.3, с. 256–260 — совершенные числа, тест Люка—Лемера;
   - §8.6, с. 338 — протокол Диффи—Хеллмана;
   - §9.1–9.3, с. 347–367 — порядок (теоремы 9.1, 9.4), число элементов порядка $d$ (теорема 9.8), гипотеза Артина (с. 357), критерий существования первообразного корня (теорема 9.15);
   - §9.4, с. 368–377 — дискретные логарифмы (индексы); §9.6, с. 385–389 — минимальный универсальный показатель $\lambda(n)$.
5. Kraft J. S., Washington L. C. *An Introduction to Number Theory with Cryptography*. — 2nd ed. — Boca Raton : CRC Press, 2018.
   - §8.1–8.2, с. 189–197 — теоремы Ферма и Эйлера, функция $\varphi$;
   - §11.1–11.2, с. 239–249 — порядок и первообразные корни, критерий (предложение 11.11); §11.5, с. 257–262 — дискретный логарифм, «шаг младенца — шаг великана»; §11.6, с. 263–266 — существование первообразных корней;
   - §12.1, с. 273–275 — протокол Диффи—Хеллмана; §14.1, с. 319 — пробное деление.
6. Herkommer M. A. *Number Theory: A Programmer's Guide*. — New York : McGraw-Hill, 1999.
   - §3.0–3.4, с. 49–69 — разложение пробным делением, $\varphi$ подсчётом НОД и через разложение, $\tau$, $\sigma$ (программы на C);
   - §4.4, с. 96–101 — порядок и первообразные корни (программа таблицы порядков).
7. Weissman M. H. *An Illustrated Theory of Numbers*. — Providence, RI : AMS, 2017; сопровождающий ноутбук P3wNT Notebook 4 — источник разделов 3.1–3.2.

### Дополнительная литература

8. Shoup V. *A Computational Introduction to Number Theory and Algebra*. — 2nd ed. — Cambridge University Press, 2009.
9. Мартынов Л. М. *Алгебра и теория чисел для криптографии* : учебное пособие. — СПб. : Лань, 2020.
10. Рябко Б. Я., Фионов А. Н. *Криптографические методы защиты информации* : учебное пособие для вузов. — М. : Горячая линия—Телеком, 2005.
11. [SageMath Reference Manual: Elementary Number Theory](https://doc.sagemath.org/html/en/reference/rings_standard/).